# Deploy VSS 3.3.0 (Video Search & Summarization)

This notebook deploys the NVIDIA VSS 3.3.0 Blueprint on GPU-equipped cloud instances.

**What it does:**
1. Validates GPU hardware and Docker prerequisites
2. Installs and configures the NGC CLI
3. Configures Docker storage for large image pulls
4. Gets the deployment code (from a local path or GitHub)
5. Detects network configuration (internal + external IPs)
6. Generates the profile's environment file and deploys with `docker compose`
7. Verifies all services are healthy

**Supported profiles:** `base`, `search`, `alerts`, `lvs`  
**Supported hardware profiles:** `H100`, `L40S`, `RTXPRO4500BW`, `RTXPRO6000BW`, `DGX-SPARK`

On `DGX-SPARK` the profiles are `base`, `alerts` and `search`, and `search` there requires a remote LLM **and** a remote VLM — the board's single GPU is fully taken by the perception pipeline.

---

## Prerequisites

- Linux instance with a supported hardware profile (`H100`, `L40S`, `RTXPRO4500BW`, `RTXPRO6000BW`, or `DGX-SPARK`)
- NVIDIA driver 580+ with CUDA 12.0+ runtime support. Exact minimum depends on platform: x86 Ubuntu 24.04 → `580.105.08`, x86 Ubuntu 22.04 → `580.65.06`, DGX Spark → `580.95.05`.
- On `DGX-SPARK`, the `sys-cache-cleaner.sh` prerequisite. Section 8 installs and starts it, and re-checks on every deploy because a reboot stops it.
- Docker Engine 28.3.3+ and <29.5.0 with Docker Compose v2.39.1+. If your launchable image starts with a newer Docker Engine, Section 4.1 pins it before deployment.
- NGC API key from [ngc.nvidia.com](https://ngc.nvidia.com)
- **500GB+ disk space** for Docker images and models. Most GPU cloud instances have a small root disk (~200-250GB) plus a large ephemeral NVMe. Section 4 will auto-detect this and move Docker/containerd storage to the NVMe.

## 1. Configuration

Set your NGC API key, deployment profile, and hardware below. These variables are used by all subsequent cells.

In [ ]:
import os

# ============================================================
# REQUIRED: Set these before running anything else
# ============================================================

NGC_CLI_API_KEY = ""          # Your NGC API key — get one at https://ngc.nvidia.com
NGC_CLI_ORG = "nvidia"        # change NGC org as needed based on your model source

PROFILE = "base"              # Deployment profile: base, search, alerts, lvs

HARDWARE_PROFILE = "RTXPRO6000BW"  # Hardware: RTXPRO6000BW, RTXPRO4500BW, H100, L40S, DGX-SPARK

# ============================================================
# OPTIONAL: Override defaults if needed
# ============================================================

# Deployment source — set ONE of these:
#   DEPLOY_SOURCE_PATH: Path to the repo cloned during launchable bringup.
#                       Must contain deploy/docker/ with compose.yml and containers.env.
#   If the default launchable path is missing, falls back to GitHub clone.
_DEFAULT_DEPLOY_SOURCE_PATH = os.path.expanduser("~/video-search-and-summarization")
DEPLOY_SOURCE_PATH = _DEFAULT_DEPLOY_SOURCE_PATH if os.path.isdir(_DEFAULT_DEPLOY_SOURCE_PATH) else ""

GIT_BRANCH = "release/3.3.0" # Git branch or tag (only used when cloning from GitHub)

ALERTS_MODE = "verification"  # Only used when PROFILE=alerts: verification or real-time

USE_REMOTE_LLM = False        # Set True to use a remote LLM endpoint instead of local
REMOTE_LLM_ENDPOINT_URL = ""  # Required when USE_REMOTE_LLM=True. Base URL without /v1, e.g. http://global.stg.ga.launchpad.nvidia.com:11571

USE_REMOTE_VLM = False        # Set True to use a remote VLM endpoint instead of local
REMOTE_VLM_ENDPOINT_URL = ""  # Required when USE_REMOTE_VLM=True. Base URL without /v1, e.g. http://global.stg.ga.launchpad.nvidia.com:11432

# Network overrides (auto-detected in Section 7 if left empty)
HOST_IP_OVERRIDE = ""         # Internal IP — leave empty for auto-detect
EXTERNAL_IP_OVERRIDE = ""     # External IP — leave empty for auto-detect

In [ ]:
# ---- Validate configuration ----
import os, sys

assert NGC_CLI_API_KEY, "NGC_CLI_API_KEY is required. Get one at https://ngc.nvidia.com"
assert PROFILE in ("base", "search", "alerts", "lvs"), f"Invalid PROFILE: {PROFILE}"
assert HARDWARE_PROFILE in ("H100", "L40S", "RTXPRO4500BW", "RTXPRO6000BW", "DGX-SPARK"), \
    f"Invalid HARDWARE_PROFILE: {HARDWARE_PROFILE}"

if PROFILE == "alerts":
    assert ALERTS_MODE in ("verification", "real-time"), f"Invalid ALERTS_MODE: {ALERTS_MODE}"

# DGX-SPARK has a single GPU. base and alerts run locally on it; search's
# perception pipeline owns the GPU outright, so both models must be remote.
if HARDWARE_PROFILE == "DGX-SPARK":
    assert PROFILE in ("base", "alerts", "search"), \
        f"{HARDWARE_PROFILE} only supports base, alerts and search profiles, not {PROFILE}"
    if PROFILE == "search":
        assert USE_REMOTE_LLM and REMOTE_LLM_ENDPOINT_URL, \
            "Search on DGX-SPARK cannot host a local LLM: set USE_REMOTE_LLM=True with REMOTE_LLM_ENDPOINT_URL"
        assert USE_REMOTE_VLM and REMOTE_VLM_ENDPOINT_URL, \
            "Search on DGX-SPARK cannot host a local VLM: set USE_REMOTE_VLM=True with REMOTE_VLM_ENDPOINT_URL"

if DEPLOY_SOURCE_PATH:
    assert os.path.isdir(DEPLOY_SOURCE_PATH), f"DEPLOY_SOURCE_PATH does not exist: {DEPLOY_SOURCE_PATH}"

if USE_REMOTE_LLM:
    assert REMOTE_LLM_ENDPOINT_URL, "REMOTE_LLM_ENDPOINT_URL is required when USE_REMOTE_LLM=True"
if USE_REMOTE_VLM:
    assert REMOTE_VLM_ENDPOINT_URL, "REMOTE_VLM_ENDPOINT_URL is required when USE_REMOTE_VLM=True"

# Export NGC key to environment for shell cells and the deployment helpers
os.environ["NGC_CLI_API_KEY"] = NGC_CLI_API_KEY
if USE_REMOTE_LLM:
    os.environ["LLM_ENDPOINT_URL"] = REMOTE_LLM_ENDPOINT_URL
if USE_REMOTE_VLM:
    os.environ["VLM_ENDPOINT_URL"] = REMOTE_VLM_ENDPOINT_URL

print("Configuration valid.")
print(f"  Profile:  {PROFILE}")
print(f"  Hardware: {HARDWARE_PROFILE}")
print(f"  Source:   {DEPLOY_SOURCE_PATH or f'GitHub (branch: {GIT_BRANCH})'}")
print(f"  NGC org:  {NGC_CLI_ORG or 'account default'}")
print(f"  LLM:      {'remote' if USE_REMOTE_LLM else 'local'}")
if USE_REMOTE_LLM:
    print(f"    endpoint: {REMOTE_LLM_ENDPOINT_URL}")
print(f"  VLM:      {'remote' if USE_REMOTE_VLM else 'local'}")
if USE_REMOTE_VLM:
    print(f"    endpoint: {REMOTE_VLM_ENDPOINT_URL}")
if PROFILE == "alerts":
    print(f"  Alerts:   {ALERTS_MODE}")
print(f"  NGC key:  {NGC_CLI_API_KEY[:4]}...{NGC_CLI_API_KEY[-4:]}")

## 1.1 Test Remote LLM / VLM Endpoints (optional)

If you enabled `USE_REMOTE_LLM` and/or `USE_REMOTE_VLM` in Section 1, run this cell to verify each endpoint **before** deploying. It probes `{ENDPOINT}/v1/models` — the same call Section 8 makes to auto-detect the model name.

An endpoint passes when:
- `GET /v1/models` returns HTTP 200 with an OpenAI-compatible `data[].id` list, and
- it advertises **exactly one** model (so the model name can be auto-detected).

Skipped automatically for local (non-remote) LLM/VLM.

In [ ]:
import json, urllib.request, urllib.error

def probe_remote_endpoint(name, base_url):
    """Probe {base_url}/v1/models — the same call Section 8 uses to auto-detect the model."""
    # Normalize: strip trailing slash and any /v1 or /v1/models the user may have added.
    url_base = base_url.rstrip("/")
    for suffix in ("/v1/models", "/v1"):
        if url_base.endswith(suffix):
            url_base = url_base[: -len(suffix)]
    url = f"{url_base}/v1/models"

    try:
        with urllib.request.urlopen(url, timeout=15) as resp:
            body = json.loads(resp.read().decode())
    except urllib.error.HTTPError as e:
        raise RuntimeError(f"{name}: HTTP {e.code} from {url}") from None
    except Exception as e:
        raise RuntimeError(f"{name}: could not reach {url}: {e}") from None

    models = [m.get("id") for m in body.get("data", []) if m.get("id")]
    if not models:
        raise RuntimeError(f"{name}: no models advertised at {url}")

    print(f"{name}: OK — {len(models)} model(s) at {url_base}")
    for m in models:
        print(f"    - {m}")
    if len(models) > 1:
        print(f"  WARNING: {name} advertises multiple models. Auto-select is unsafe;")
        print(f"           use an endpoint that serves a single model, or a model-specific URL.")
    return models

if USE_REMOTE_LLM:
    probe_remote_endpoint("LLM", REMOTE_LLM_ENDPOINT_URL)
else:
    print("LLM: local — skipping remote probe (verify after deploy in Section 9).")

if USE_REMOTE_VLM:
    probe_remote_endpoint("VLM", REMOTE_VLM_ENDPOINT_URL)
else:
    print("VLM: local — skipping remote probe (verify after deploy in Section 9).")

print("\nEndpoint check complete.")

## 2. Prerequisites Check

Validate that the NVIDIA driver, CUDA, Docker, Docker Compose, and NVIDIA container runtime meet the deployment prerequisites. Docker versions newer than the supported range are reported here, then pinned in Section 4.1 before deployment.

In [ ]:
%%bash
set -euo pipefail

set_min_driver_version() {
    # Documented minimum NVIDIA driver by OS / platform:
    #   x86 Ubuntu 24.04 -> 580.105.08 | x86 Ubuntu 22.04 -> 580.65.06
    #   DGX Spark        -> 580.95.05
    local os_version_id="" product_name="" gpu_name="" platform_info=""

    if [ -r /etc/os-release ]; then
        os_version_id=$(. /etc/os-release; echo "${VERSION_ID:-}")
    fi

    for f in /sys/devices/virtual/dmi/id/product_name /proc/device-tree/model; do
        if [ -r "$f" ]; then
            product_name=$(tr -d '\0' < "$f" 2>/dev/null)
            [ -n "$product_name" ] && break
        fi
    done

    if command -v nvidia-smi >/dev/null 2>&1; then
        gpu_name=$(nvidia-smi --query-gpu=name --format=csv,noheader 2>/dev/null | head -n1)
    fi
    platform_info="$product_name $gpu_name"

    if echo "$platform_info" | grep -qiE 'spark'; then
        MIN_DRIVER_VERSION="580.95.05"
    elif [ "$os_version_id" = "24.04" ]; then
        MIN_DRIVER_VERSION="580.105.08"
    elif [ "$os_version_id" = "22.04" ]; then
        MIN_DRIVER_VERSION="580.65.06"
    else
        MIN_DRIVER_VERSION="580.65.06"
    fi
}
set_min_driver_version

MIN_CUDA_VERSION="12.0"
MIN_DOCKER_VERSION="28.3.3"
MAX_DOCKER_VERSION="29.5.0"
MIN_COMPOSE_VERSION="2.39.1"

fail() {
    echo "ERROR: $*" >&2
    exit 1
}

require_command() {
    command -v "$1" >/dev/null 2>&1 || fail "$1 is not installed or not on PATH"
}

version_ge() {
    [ "$(printf '%s\n%s\n' "$2" "$1" | sort -V | head -n1)" = "$2" ]
}

version_lt() {
    [ "$1" != "$2" ] && [ "$(printf '%s\n%s\n' "$1" "$2" | sort -V | head -n1)" = "$1" ]
}

echo "=== NVIDIA Driver & GPU ==="
require_command nvidia-smi
nvidia-smi --query-gpu=index,name,driver_version,memory.total --format=csv,noheader

GPU_COUNT=$(nvidia-smi --query-gpu=index --format=csv,noheader | wc -l | tr -d ' ')
[ "$GPU_COUNT" -gt 0 ] || fail "No NVIDIA GPUs detected"
echo "Detected $GPU_COUNT GPU(s)"

DRIVER_VERSION=$(nvidia-smi --query-gpu=driver_version --format=csv,noheader | head -n1 | tr -d ' ')
version_ge "$DRIVER_VERSION" "$MIN_DRIVER_VERSION" \
    || fail "NVIDIA driver $DRIVER_VERSION is older than required $MIN_DRIVER_VERSION"
echo "NVIDIA driver: OK ($DRIVER_VERSION >= $MIN_DRIVER_VERSION)"

CUDA_VERSION=$(nvidia-smi | awk -F'CUDA Version: ' '/CUDA Version/ {split($2,a," "); version=a[1]} END {print version}')
[ -n "$CUDA_VERSION" ] || fail "Could not read CUDA version from nvidia-smi"
version_ge "$CUDA_VERSION" "$MIN_CUDA_VERSION" \
    || fail "CUDA version $CUDA_VERSION is older than required $MIN_CUDA_VERSION"
echo "CUDA version: OK ($CUDA_VERSION >= $MIN_CUDA_VERSION)"
echo ""

echo "=== Docker ==="
require_command docker
docker ps >/dev/null 2>&1 \
    || fail "Docker daemon is not reachable by this user. Add the user to the docker group or start Docker."

DOCKER_VERSION=$(docker version --format '{{.Server.Version}}')
version_ge "$DOCKER_VERSION" "$MIN_DOCKER_VERSION" \
    || fail "Docker Engine $DOCKER_VERSION is older than required $MIN_DOCKER_VERSION"
if version_lt "$DOCKER_VERSION" "$MAX_DOCKER_VERSION"; then
    echo "Docker Engine: OK ($DOCKER_VERSION >= $MIN_DOCKER_VERSION and < $MAX_DOCKER_VERSION)"
else
    echo "WARNING: Docker Engine $DOCKER_VERSION is newer than the supported range (< $MAX_DOCKER_VERSION)."
    echo "Section 4.1 will pin Docker before deployment image pulls. Continue through the notebook in order."
fi

COMPOSE_VERSION=$(docker compose version --short 2>/dev/null | sed 's/^v//')
[ -n "$COMPOSE_VERSION" ] || fail "Docker Compose plugin is not installed"
version_ge "$COMPOSE_VERSION" "$MIN_COMPOSE_VERSION" \
    || fail "Docker Compose $COMPOSE_VERSION is older than required $MIN_COMPOSE_VERSION"
echo "Docker Compose: OK ($COMPOSE_VERSION >= $MIN_COMPOSE_VERSION)"
echo ""

echo "=== NVIDIA Container Toolkit ==="
docker run --rm --gpus all nvidia/cuda:12.0.0-base-ubuntu22.04 nvidia-smi >/dev/null 2>&1 \
    || fail "NVIDIA Container Toolkit is not functional. Install or repair it: https://docs.nvidia.com/datacenter/cloud-native/container-toolkit/latest/install-guide.html"
echo "NVIDIA Container Toolkit: OK"
echo ""

echo "=== Disk Space ==="
df -h / | tail -1 | awk '{print "Root:", $4, "available of", $2}'
echo ""
echo "Prerequisites check passed."

## 3. Install NGC CLI

The NGC CLI is required to download models during deployment. This cell installs it if not already present, then configures it with your API key.

In [ ]:
import subprocess, os, shutil

def run(cmd, **kwargs):
    """Run a shell command, raise on failure with output."""
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, **kwargs)
    if r.returncode != 0:
        raise RuntimeError(f"Command failed: {cmd}\n{r.stderr}\n{r.stdout}")
    return r.stdout.strip()

# Check if NGC CLI is already installed
ngc_path = shutil.which("ngc")
if ngc_path:
    ver = run("ngc --version 2>&1 | head -1")
    print(f"NGC CLI already installed: {ver}")
else:
    import platform
    arch = platform.machine()
    if arch in ("aarch64", "arm64"):
        filename = "ngccli_linux_arm64.zip"
    else:
        filename = "ngccli_linux.zip"

    # Use version-pinned URL (update this if a newer version is needed)
    NGC_CLI_VERSION = "4.13.0"
    url = f"https://api.ngc.nvidia.com/v2/resources/nvidia/ngc-apps/ngc_cli/versions/{NGC_CLI_VERSION}/files/{filename}"

    print(f"Installing NGC CLI {NGC_CLI_VERSION} ...")
    run(f"cd /tmp && wget -q --content-disposition '{url}' -O ngc_cli.zip")

    # Verify download is not empty
    size = os.path.getsize("/tmp/ngc_cli.zip")
    if size < 1000:
        raise RuntimeError(f"NGC CLI download failed — file is only {size} bytes. Check the version URL.")
    print(f"  Downloaded {size / 1024 / 1024:.1f} MB")

    run("cd /tmp && unzip -o ngc_cli.zip")
    # NGC bundles its own Python — copy the entire directory
    run("sudo cp -r /tmp/ngc-cli/* /usr/local/bin/")
    run("rm -rf /tmp/ngc_cli.zip /tmp/ngc-cli")

    ver = run("ngc --version 2>&1 | head -1")
    print(f"  Installed: {ver}")

# Configure NGC CLI with API key. Leave org unset unless the user supplied one.
print("Configuring NGC CLI...")
ngc_dir = os.path.expanduser("~/.ngc")
os.makedirs(ngc_dir, exist_ok=True)

config_lines = [
    ";WARNING - This is a machine generated file. Do not edit manually.\n",
    ";WARNING - To update local config settings, see 'ngc config set -h'.\n",
    "\n",
    "[CURRENT]\n",
    f"apikey = {NGC_CLI_API_KEY}\n",
    "format_type = ascii\n",
]
if NGC_CLI_ORG:
    config_lines.append(f"org = {NGC_CLI_ORG}\n")

with open(os.path.join(ngc_dir, "config"), "w") as f:
    f.writelines(config_lines)

print("NGC CLI configured.")
print(run("ngc config current"))

## 4. Docker & Containerd Storage

Docker images and containerd layers for VSS require **~250GB** (NIM models, DeepStream, ELK, etc.). Most GPU cloud instances ship with a small root disk (200-250GB) that **will run out of space** during deployment.

This cell auto-detects whether your root disk is too small and moves Docker and containerd storage to a larger mount. Docker **volumes** (Elasticsearch indices, uploaded videos, Kafka data) are kept on the root disk so your data persists even if the instance is stopped and the ephemeral NVMe is wiped. Images and layers are re-pulled automatically on next deploy.

**Common NVMe mount points** (auto-detected):
- AWS DLAMI: `/opt/dlami/nvme`
- Brev/Crusoe: `/ephemeral`
- Custom RAID: `/data`

To override auto-detection, set `STORAGE_ROOT` below.

In [ ]:
import subprocess, json, os, shutil, tempfile

STORAGE_ROOT = ""  # Override: set to a mount path (e.g. "/mnt/data") to skip auto-detection

MIN_ROOT_FREE_GB = 350  # If root has less than this free, move storage

# --- Auto-detect large mount ---

def get_disk_free_gb(path):
    """Return free space in GB for the filesystem containing path."""
    st = os.statvfs(path)
    return (st.f_bavail * st.f_frsize) / (1024 ** 3)

def get_disk_total_gb(path):
    st = os.statvfs(path)
    return (st.f_blocks * st.f_frsize) / (1024 ** 3)

def find_large_mount():
    """Look for a large non-root mount suitable for Docker storage."""
    candidates = ["/opt/dlami/nvme", "/ephemeral", "/data"]
    for path in candidates:
        if os.path.isdir(path) and os.path.ismount(path):
            free = get_disk_free_gb(path)
            if free > 200:
                return path, free
    return None, 0

def find_mount_unit(mount_path):
    """Convert a mount path to a systemd mount unit name (e.g. /opt/dlami/nvme -> opt-dlami-nvme.mount)."""
    # Strip leading slash, replace remaining slashes with dashes
    unit = mount_path.strip("/").replace("/", "-") + ".mount"
    # Verify this unit exists on the system
    r = subprocess.run(["systemctl", "cat", unit], capture_output=True, text=True)
    if r.returncode == 0:
        return unit
    return None

def get_docker_engine_version():
    """Return the Docker Engine server version, or an empty string if unavailable."""
    r = subprocess.run(["docker", "version", "--format", "{{.Server.Version}}"],
                       capture_output=True, text=True)
    if r.returncode == 0:
        return r.stdout.strip()

    print("Unable to determine Docker Engine server version; skipping Docker 29.5.x/NVCR workaround.")
    return ""

def read_daemon_config(daemon_json):
    try:
        with open(daemon_json) as f:
            return json.load(f)
    except (FileNotFoundError, json.JSONDecodeError):
        return {}

def write_daemon_config(daemon_json, config):
    with tempfile.NamedTemporaryFile(mode="w", suffix=".json", delete=False) as tmp:
        json.dump(config, tmp, indent=2)
        tmp.write("\n")
        tmp_path = tmp.name
    try:
        subprocess.run(["sudo", "cp", tmp_path, daemon_json], check=True)
    finally:
        os.unlink(tmp_path)

def prepare_docker_295_nvcr_workaround(config, docker_version):
    """Disable Docker's containerd image store for Docker 29.5.x NVCR pulls."""
    if not docker_version.startswith("29.5."):
        return False

    print(f"\nApplying temporary Docker 29.5.x/NVCR workaround for Docker {docker_version}.")
    print("  Setting daemon feature containerd-snapshotter=false to avoid NVCR Incorrect Repository Format pulls.")

    features = config.get("features")
    if not isinstance(features, dict):
        features = {}
        config["features"] = features

    if features.get("containerd-snapshotter") is False:
        print("  Docker 29.5.x/NVCR workaround already configured; no Docker restart needed for this patch.")
        return False

    # Temporary Docker 29.5.x/NVCR workaround: Docker 29.5.x with the
    # containerd image store can fail some nvcr.io pulls with
    # "Incorrect Repository Format". Remove this once Docker/NVCR resolves it.
    features["containerd-snapshotter"] = False
    return True

daemon_json = "/etc/docker/daemon.json"
daemon_config = read_daemon_config(daemon_json)
docker_engine_version = get_docker_engine_version()
need_docker_295_daemon_json = prepare_docker_295_nvcr_workaround(daemon_config, docker_engine_version)

root_free = get_disk_free_gb("/")
root_total = get_disk_total_gb("/")

print(f"Root disk: {root_free:.0f} GB free / {root_total:.0f} GB total")

if STORAGE_ROOT:
    large_mount = STORAGE_ROOT
    mount_free = get_disk_free_gb(STORAGE_ROOT)
    print(f"Using override: {STORAGE_ROOT} ({mount_free:.0f} GB free)")
    need_move = True
else:
    large_mount, mount_free = find_large_mount()
    need_move = root_free < MIN_ROOT_FREE_GB and large_mount is not None

    if large_mount:
        print(f"Large mount:    {large_mount} ({mount_free:.0f} GB free)")
    else:
        print("No large ephemeral mount detected.")

    if root_free >= MIN_ROOT_FREE_GB:
        print(f"\nRoot disk has enough space ({root_free:.0f} GB free). No storage move needed.")
    elif not large_mount:
        print(f"\nWARNING: Root disk only has {root_free:.0f} GB free and no large mount was found.")
        print("Deployment may fail due to disk space. Consider attaching a larger volume.")

if need_move:
    DOCKER_DATA_ROOT = os.path.join(large_mount, "docker")
    CONTAINERD_ROOT = os.path.join(large_mount, "containerd")
    VOLUMES_DIR = "/var/lib/docker/volumes"  # Keep volumes on persistent root disk

    print(f"\nMoving Docker and containerd storage to {large_mount}")
    print(f"  Docker images/layers: {DOCKER_DATA_ROOT}")
    print(f"  Containerd:           {CONTAINERD_ROOT}")
    print(f"  Docker volumes:       {VOLUMES_DIR} (stays on root for persistence)")

    # --- Check what needs changing ---
    config = daemon_config

    need_data_root_daemon_json = config.get("data-root") != DOCKER_DATA_ROOT
    need_daemon_json = need_data_root_daemon_json or need_docker_295_daemon_json

    subprocess.run(["sudo", "mkdir", "-p", DOCKER_DATA_ROOT], check=True)
    subprocess.run(["sudo", "mkdir", "-p", VOLUMES_DIR], check=True)

    volumes_link = os.path.join(DOCKER_DATA_ROOT, "volumes")
    need_volumes_symlink = not (os.path.islink(volumes_link) and os.readlink(volumes_link) == VOLUMES_DIR)

    containerd_link = "/var/lib/containerd"
    need_containerd = not (os.path.islink(containerd_link) and os.readlink(containerd_link) == CONTAINERD_ROOT)

    # Even if symlinks are correct, ensure NVMe target dirs actually exist
    # (they get wiped when ephemeral NVMe is reset on instance stop/start)
    need_target_dirs = not os.path.isdir(DOCKER_DATA_ROOT) or not os.path.isdir(CONTAINERD_ROOT)
    if need_target_dirs:
        print(f"\n  NVMe target dir(s) missing — recreating...")
        subprocess.run(["sudo", "mkdir", "-p", DOCKER_DATA_ROOT, CONTAINERD_ROOT], check=True)

    if not need_daemon_json and not need_volumes_symlink and not need_containerd:
        print(f"\n  Docker data-root already set to {DOCKER_DATA_ROOT}")
        print(f"  Volumes symlink already correct: {volumes_link} -> {VOLUMES_DIR}")
        print(f"  Containerd already symlinked: {containerd_link} -> {CONTAINERD_ROOT}")

        # Always ensure the boot-time restore service is up to date
        # (handles the case where service exists but is missing mount dependencies)
        _update_restore_service = True
        _need_restart = need_target_dirs  # Restart Docker/containerd if we had to recreate dirs
    else:
        _update_restore_service = True
        _need_restart = True

        # Stop Docker AND docker.socket (socket can reactivate Docker and recreate dirs)
        print("\n  Stopping Docker and containerd for storage reconfiguration...")
        subprocess.run(["sudo", "systemctl", "stop", "docker.socket"], check=False)
        subprocess.run(["sudo", "systemctl", "stop", "docker"], check=True)
        subprocess.run(["sudo", "systemctl", "stop", "containerd"], check=True)

        # --- Docker daemon.json ---
        if need_daemon_json:
            if need_data_root_daemon_json:
                config["data-root"] = DOCKER_DATA_ROOT
            write_daemon_config(daemon_json, config)
            if need_docker_295_daemon_json:
                print("  Docker 29.5.x/NVCR workaround written to daemon.json")
            if need_data_root_daemon_json:
                print(f"  Docker data-root set to {DOCKER_DATA_ROOT}")
            else:
                print(f"  Docker data-root already set to {DOCKER_DATA_ROOT}")
        else:
            print(f"  Docker data-root already set to {DOCKER_DATA_ROOT}")

        # --- Volumes symlink (use ln -sfn for idempotency) ---
        if need_volumes_symlink:
            # ln -sfn: force, no-dereference (replaces existing dir/symlink atomically)
            subprocess.run(["sudo", "rm", "-rf", volumes_link], check=True)
            subprocess.run(["sudo", "ln", "-sfn", VOLUMES_DIR, volumes_link], check=True)
            print(f"  Created symlink: {volumes_link} -> {VOLUMES_DIR}")
        else:
            print(f"  Volumes symlink already correct: {volumes_link} -> {VOLUMES_DIR}")

        # --- Containerd ---
        if need_containerd:
            subprocess.run(["sudo", "mkdir", "-p", CONTAINERD_ROOT], check=True)
            if os.path.isdir(containerd_link) and not os.path.islink(containerd_link):
                # Move existing containerd data
                subprocess.run(f"sudo mv {containerd_link}/* {CONTAINERD_ROOT}/ 2>/dev/null; true",
                               shell=True, check=False)
                subprocess.run(["sudo", "rm", "-rf", containerd_link], check=True)
                print(f"  Containerd data moved to {CONTAINERD_ROOT}")
            elif os.path.lexists(containerd_link):
                subprocess.run(["sudo", "rm", "-f", containerd_link], check=True)
            subprocess.run(["sudo", "ln", "-sfn", CONTAINERD_ROOT, containerd_link], check=True)
            print(f"  Containerd symlinked: {containerd_link} -> {CONTAINERD_ROOT}")
        else:
            print(f"  Containerd already symlinked: {containerd_link} -> {CONTAINERD_ROOT}")

    # --- Install/update boot-time restore service ---
    # Ephemeral NVMe is wiped on instance stop/start. This systemd service
    # recreates the directories before Docker/containerd start so they don't crash-loop.
    # It also re-creates the volumes symlink: without it Docker would populate a
    # fresh volumes directory on the NVMe and stop seeing the data kept on root.
    # We use RequiresMountsFor= so the service waits for the NVMe to actually be mounted.
    if _update_restore_service:
        unit_name = "docker-nvme-restore.service"
        unit_path = f"/etc/systemd/system/{unit_name}"

        # Build After= line — include the mount unit if systemd knows about it
        after_targets = "local-fs.target"
        mount_unit = find_mount_unit(large_mount)
        if mount_unit:
            after_targets += f" {mount_unit}"

        unit_content = f"""[Unit]
Description=Restore Docker/containerd dirs on ephemeral NVMe
Before=containerd.service docker.service
After={after_targets}
RequiresMountsFor={large_mount}

[Service]
Type=oneshot
ExecStart=/bin/bash -c 'mkdir -p {DOCKER_DATA_ROOT} {CONTAINERD_ROOT} {VOLUMES_DIR} && if [ ! -e {DOCKER_DATA_ROOT}/volumes ] || [ -L {DOCKER_DATA_ROOT}/volumes ]; then ln -sfn {VOLUMES_DIR} {DOCKER_DATA_ROOT}/volumes; fi'

[Install]
WantedBy=multi-user.target
"""
        import tempfile
        with tempfile.NamedTemporaryFile(mode='w', suffix='.service', delete=False) as tmp:
            tmp.write(unit_content)
            tmp_path = tmp.name
        subprocess.run(["sudo", "cp", tmp_path, unit_path], check=True)
        os.unlink(tmp_path)
        subprocess.run(["sudo", "systemctl", "daemon-reload"], check=True)
        subprocess.run(["sudo", "systemctl", "enable", unit_name], check=True, capture_output=True)
        print(f"  Installed {unit_name} (restores NVMe dirs on boot, waits for mount)")

    # --- Restart if needed ---
    if _need_restart:
        print("\n  Starting containerd and Docker...")
        subprocess.run(["sudo", "systemctl", "start", "containerd"], check=True)
        subprocess.run(["sudo", "systemctl", "start", "docker.socket"], check=True)
        subprocess.run(["sudo", "systemctl", "start", "docker"], check=True)

    r = subprocess.run(["docker", "info", "--format", "{{.DockerRootDir}}"],
                       capture_output=True, text=True)
    print(f"\n  Docker data-root: {r.stdout.strip()}")
    target = os.readlink(containerd_link) if os.path.islink(containerd_link) else containerd_link
    print(f"  Containerd root:  {target}")
    print(f"\n  Storage configuration complete.")
else:
    if need_docker_295_daemon_json:
        write_daemon_config(daemon_json, daemon_config)
        print("  Docker 29.5.x/NVCR workaround written to daemon.json")
        print("\n  Restarting Docker to activate daemon.json change...")
        subprocess.run(["sudo", "systemctl", "restart", "docker"], check=True)
    if not STORAGE_ROOT and root_free >= MIN_ROOT_FREE_GB:
        print("Skipping storage move.")

### 4.1 Pin Docker version

Pin Docker CE + plugins + containerd.io to a known-good combination (CE **29.4.3**, buildx **0.33.0**, compose **5.1.3**, containerd **2.2.3**). Some Brev launchables ship newer versions than the VSS deploy profiles are tested against; pin explicitly so compose/buildx incompatibilities don't surface in later sections. `apt-mark hold` prevents unattended-upgrades or later cells from drifting the box back.

The cell first reads the installed Docker Engine version: if it already falls in the tested range **[28.3.3, 29.5.0)** (the same range Section 3 validates) the version downgrade is **skipped** — re-pinning to an exact version the platform's apt repo may not carry (e.g. DGX Spark / DGX-OS on arm64) would fail with *version not found* for no benefit. The packages are still `apt-mark hold`-ed at their current versions so the box can't drift past the tested range mid-run. Safe to re-run.

Runs *after* the section 4 storage relocation so the APT download lands on the relocated volume and the dockerd restart triggered by the downgrade picks up the new data-root.

In [ ]:
%%bash
# Pin Docker CE + plugins + containerd.io to a known-good combination, but
# ONLY when the host's Docker is outside the tested range. Some Brev
# launchables ship a newer Docker than the VSS deploy profiles are tested
# against; pin explicitly so compose/buildx incompatibilities don't surface
# mid-deployment.
#
# When the installed Docker already falls in [28.3.3, 29.5.0) the version
# downgrade is skipped: re-pinning to an exact epoch-versioned package that
# the platform's apt repo may not carry (e.g. DGX Spark / DGX-OS on arm64)
# fails with "version not found" for no benefit. The in-range packages are
# still held so the box can't drift past the tested range mid-notebook.
# Idempotent -- safe to re-run.

set -euo pipefail

# Tested Docker Engine range -- keep in sync with the VSS launchable prereq check.
MIN_DOCKER_VERSION="28.3.3"
MAX_DOCKER_VERSION="29.5.0"

# Packages frozen with `apt-mark hold` so unattended-upgrades / later
# `apt-get install` calls can't drift the box before the notebook finishes.
HOLD_PKGS="docker-ce docker-ce-cli docker-buildx-plugin docker-compose-plugin containerd.io"

version_ge() { [ "$(printf '%s\n%s\n' "$2" "$1" | sort -V | head -n1)" = "$2" ]; }
version_lt() { [ "$1" != "$2" ] && [ "$(printf '%s\n%s\n' "$1" "$2" | sort -V | head -n1)" = "$1" ]; }

DOCKER_VERSION="$(docker version --format '{{.Server.Version}}' 2>/dev/null || true)"
if [ -n "$DOCKER_VERSION" ] \
   && version_ge "$DOCKER_VERSION" "$MIN_DOCKER_VERSION" \
   && version_lt "$DOCKER_VERSION" "$MAX_DOCKER_VERSION"; then
  echo "Docker $DOCKER_VERSION is within the tested range [$MIN_DOCKER_VERSION, $MAX_DOCKER_VERSION); skipping the Docker version pin."
  # No downgrade needed, but still hold the in-range packages at their
  # current versions so unattended-upgrades / later apt-get calls can't drift
  # the box past the tested range for the remainder of the notebook.
  sudo apt-mark hold $HOLD_PKGS
  exit 0
fi

if [ -n "$DOCKER_VERSION" ]; then
  echo "Docker $DOCKER_VERSION is outside the tested range [$MIN_DOCKER_VERSION, $MAX_DOCKER_VERSION); pinning to known-good versions."
else
  echo "Could not read the installed Docker version; pinning to known-good versions."
fi

# Read distro info from /etc/os-release (always present on Ubuntu; minimal
# images don't ship `lsb_release`).
. /etc/os-release
DISTRO="${VERSION_ID}"
CODENAME="${UBUNTU_CODENAME:-${VERSION_CODENAME}}"

# Versions hard-coded to what shipped alongside docker-ce 29.4.3 on the
# Docker apt repo (verified against download.docker.com + upstream GitHub
# release timestamps). When bumping DOCKER_CE_VER, bump these four together.
DOCKER_CE_VER="5:29.4.3-1~ubuntu.${DISTRO}~${CODENAME}"
BUILDX_VER="0.33.0-1~ubuntu.${DISTRO}~${CODENAME}"
COMPOSE_VER="5.1.3-1~ubuntu.${DISTRO}~${CODENAME}"
CONTAINERD_VER="2.2.3-1~ubuntu.${DISTRO}~${CODENAME}"

# Refresh the APT cache first -- without this, the specific epoch-versioned
# package may not be in the local index and the install would fail with
# version-not-found before any pinning takes effect.
sudo apt-get update -qq

sudo DEBIAN_FRONTEND=noninteractive apt-get install -y \
  --allow-downgrades \
  -o Dpkg::Options::=--force-confdef \
  -o Dpkg::Options::=--force-confold \
  docker-ce="$DOCKER_CE_VER" \
  docker-ce-cli="$DOCKER_CE_VER" \
  docker-buildx-plugin="$BUILDX_VER" \
  docker-compose-plugin="$COMPOSE_VER" \
  containerd.io="$CONTAINERD_VER"

# Hold so unattended-upgrades / later `apt-get install` calls don't drift
# the box back to newer versions before the rest of the notebook runs.
sudo apt-mark hold $HOLD_PKGS

## 5. Docker Login

Authenticate with the NVIDIA Container Registry (`nvcr.io`) to pull deployment images.

In [ ]:
import subprocess

result = subprocess.run(
    ["docker", "login", "nvcr.io",
     "--username", "$oauthtoken",
     "--password", NGC_CLI_API_KEY],
    capture_output=True, text=True
)
if result.returncode == 0:
    print("Docker login to nvcr.io: OK")
else:
    print(f"Docker login FAILED:\n{result.stderr}")
    raise RuntimeError("Docker login to nvcr.io failed")

## 6. Get Deployment Code

This cell locates the deployment code (scripts, compose files, configs). There are two ways to get it onto the server:

### Option 1 — Brev Launchable (automatic)

When configured as a Brev Launchable, the git repository is cloned onto the instance automatically. Set `DEPLOY_SOURCE_PATH` in Section 1 to the path where Brev placed it (typically `~/video-search-and-summarization`).

### Option 2 — Manual tarball

If the code isn't already on the server, create a tarball from your local checkout and copy it over:

```bash
# On your local machine:
cd /path/to/video-search-and-summarization
tar czf ~/vss-deploy.tar.gz --exclude='.git' .
scp ~/vss-deploy.tar.gz <user>@<server>:~/

# On the server:
mkdir -p ~/video-search-and-summarization
tar xzf ~/vss-deploy.tar.gz -C ~/video-search-and-summarization
```

Then set in **Section 1**: `DEPLOY_SOURCE_PATH = "/home/<user>/video-search-and-summarization"`

---

If `DEPLOY_SOURCE_PATH` is set, uses the code at that path directly. Otherwise, attempts to clone from GitHub (requires the repo to be accessible).

In [ ]:
import subprocess, os

if DEPLOY_SOURCE_PATH:
    # --- Use pre-extracted local repo ---
    REPO_DIR = DEPLOY_SOURCE_PATH
    print(f"Using local deployment source: {REPO_DIR}")
else:
    # --- Clone from GitHub ---
    DEPLOY_DIR = os.path.expanduser("~/deployments")
    REPO_DIR = os.path.join(DEPLOY_DIR, "video-search-and-summarization")
    GITHUB_REPO = "https://github.com/NVIDIA-AI-Blueprints/video-search-and-summarization.git"

    os.makedirs(DEPLOY_DIR, exist_ok=True)

    if os.path.isdir(REPO_DIR):
        print(f"Repo already exists at {REPO_DIR}")
        print(f"Fetching latest and checking out {GIT_BRANCH}...")
        subprocess.run(["git", "fetch", "--all", "--prune"], cwd=REPO_DIR, check=True,
                       capture_output=True)
        result = subprocess.run(["git", "checkout", GIT_BRANCH], cwd=REPO_DIR,
                               capture_output=True, text=True)
        if result.returncode != 0:
            raise RuntimeError(f"Failed to checkout {GIT_BRANCH}:\n{result.stderr}")
        subprocess.run(["git", "pull", "--ff-only"], cwd=REPO_DIR, check=False,
                       capture_output=True)
    else:
        print(f"Cloning from GitHub (branch: {GIT_BRANCH})...")
        result = subprocess.run(
            ["git", "clone", "--branch", GIT_BRANCH, "--single-branch", GITHUB_REPO, REPO_DIR],
            capture_output=True, text=True
        )
        if result.returncode != 0:
            raise RuntimeError(f"Clone failed:\n{result.stderr}")
        print("Clone complete.")

# Validate repo structure
SCRIPT_DIR = os.path.join(REPO_DIR, "deploy/docker/scripts")
assert os.path.isdir(os.path.join(REPO_DIR, "deploy/docker")), \
    f"deploy/docker/ not found in {REPO_DIR}"
for _required in ("compose.yml", "containers.env", "developer-profiles"):
    assert os.path.exists(os.path.join(REPO_DIR, "deploy/docker", _required)), \
        f"{_required} not found in {REPO_DIR}/deploy/docker"

# Show commit info (if it's a git repo)
commit = "(not a git repo)"
branch = ""
if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    commit = subprocess.run(
        ["git", "log", "--oneline", "-1"],
        cwd=REPO_DIR, capture_output=True, text=True
    ).stdout.strip()
    branch = subprocess.run(
        ["git", "branch", "--show-current"],
        cwd=REPO_DIR, capture_output=True, text=True
    ).stdout.strip()

print(f"\nRepo:    {REPO_DIR}")
if branch:
    print(f"Branch:  {branch}")
print(f"Commit:  {commit}")
print(f"Scripts: {SCRIPT_DIR}")
print(f"\nContents of deploy/docker/:")
for entry in sorted(os.listdir(os.path.join(REPO_DIR, "deploy/docker"))):
    print(f"  {entry}")

## 7. Detect Network Configuration

Auto-detects internal (`HOST_IP`) and external (`EXTERNAL_IP`) addresses. On NAT'd cloud instances (Brev, AWS), these are different — the internal IP is used for inter-container communication while the external IP is used for browser access.

If auto-detection fails or gives the wrong result, set the overrides in Section 1.

In [ ]:
import subprocess, os

def detect_internal_ip():
    """Detect internal IP via ip route."""
    try:
        out = subprocess.run(
            ["bash", "-c", "ip route get 1.1.1.1 | awk '/src/ {for (i=1;i<=NF;i++) if ($i==\"src\") print $(i+1)}'"],
            capture_output=True, text=True, timeout=5
        )
        return out.stdout.strip()
    except Exception:
        return ""

def detect_external_ip():
    """Detect external IP via public service."""
    for cmd in ["curl -s --max-time 5 ifconfig.me", "curl -s --max-time 5 icanhazip.com"]:
        try:
            out = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=10)
            ip = out.stdout.strip()
            if ip:
                return ip
        except Exception:
            continue
    return ""

def read_etc_environment():
    """Read key=value pairs from /etc/environment (Brev sets BREV_ENV_ID there)."""
    env = {}
    try:
        with open("/etc/environment") as f:
            for line in f:
                line = line.strip()
                if "=" in line and not line.startswith("#"):
                    key, _, value = line.partition("=")
                    env[key.strip()] = value.strip().strip('"')
    except FileNotFoundError:
        pass
    return env

def detect_brev_link_domain():
    """Select an override or detect Brev's Skybridge-managed NetBird network."""
    explicit_domain = os.environ.get("BREV_LINK_DOMAIN", "").strip()
    if explicit_domain:
        return explicit_domain

    try:
        netbird_status = subprocess.run(
            ["netbird", "status", "-d"],
            capture_output=True, text=True, timeout=3
        )
        status_output = f"{netbird_status.stdout or ''}\n{netbird_status.stderr or ''}".lower()
        skybridge_markers = ("skybridge", "brev.nvidia.com", "brev.dev")
        if netbird_status.returncode == 0 and any(
            marker in status_output for marker in skybridge_markers
        ):
            return "apps.run.brev.nvidia.com"
    except (OSError, subprocess.SubprocessError):
        pass

    return "brevlab.com"

HOST_IP = HOST_IP_OVERRIDE or detect_internal_ip()
EXTERNAL_IP = EXTERNAL_IP_OVERRIDE or detect_external_ip()

print(f"Internal IP (HOST_IP):   {HOST_IP}")
print(f"External IP:             {EXTERNAL_IP}")

if HOST_IP == EXTERNAL_IP:
    print("\nInternal == External (direct connection, no NAT)")
else:
    print("\nNAT detected — internal and external IPs differ.")
    print("Section 8 will set EXTERNAL_IP automatically.")

if not HOST_IP:
    print("\nWARNING: Could not detect internal IP. Set HOST_IP_OVERRIDE in Section 1.")
if not EXTERNAL_IP:
    print("\nWARNING: Could not detect external IP. Set EXTERNAL_IP_OVERRIDE in Section 1.")

# --- Brev Secure Links ---
# On Brev, all browser-facing traffic routes through the HAProxy ingress
# on a single port (default 7777). This avoids cross-origin issues when each
# port gets its own secure-link hostname.
# Check os.environ first, then fall back to /etc/environment (Jupyter kernels
# may not inherit /etc/environment depending on how the notebook server starts).
_etc_env = read_etc_environment()
BREV_ENV_ID = os.environ.get("BREV_ENV_ID") or _etc_env.get("BREV_ENV_ID", "")
if BREV_ENV_ID:
    # Ensure Brev settings are in os.environ so Section 7.1 picks them up.
    os.environ["BREV_ENV_ID"] = BREV_ENV_ID
    explicit_brev_link_domain = os.environ.get("BREV_LINK_DOMAIN", "").strip()
    brev_link_domain = detect_brev_link_domain()
    os.environ["BREV_LINK_DOMAIN"] = brev_link_domain
    if explicit_brev_link_domain:
        brev_link_provider = "explicit override"
    elif brev_link_domain == "apps.run.brev.nvidia.com":
        brev_link_provider = "Skybridge (NetBird detected)"
    else:
        brev_link_provider = "Cloudflare (NetBird unavailable)"
    proxy_port = os.environ.get("PROXY_PORT", "7777")
    # Brev secure links prefix the HAProxy port directly. Set BREV_LINK_PREFIX
    # to override if your setup differs.
    brev_link_prefix = os.environ.get("BREV_LINK_PREFIX", f"{proxy_port}")
    os.environ["BREV_LINK_PREFIX"] = brev_link_prefix
    brev_ui_url = f"https://{brev_link_prefix}-{BREV_ENV_ID}.{brev_link_domain}"
    print(f"\n=== Brev Environment Detected ===")
    print(f"  BREV_ENV_ID: {BREV_ENV_ID}")
    print(f"  Secure link provider: {brev_link_provider}")
    print(f"  Secure link domain: {brev_link_domain}")
    print(f"  Secure link prefix: {brev_link_prefix} (set BREV_LINK_PREFIX to override)")
    print(f"  All browser-facing URLs route through HAProxy ingress (port {proxy_port})")
    print(f"  UI will be available at: {brev_ui_url}")
else:
    BREV_ENV_ID = ""  # ensure defined for later cells

## 7.1 Deployment Helpers

Run this cell once before deploying. It defines the deployment logic the notebook uses in Sections 8, 13 and 14 — generating the profile's `generated.env`, creating the data directories, and driving `docker compose` directly.

Nothing here needs editing. The values you can change live in Section 1, plus the profile's `overrides.env` under `deploy/docker/developer-profiles/dev-profile-<PROFILE>/` if you want to adjust something this notebook does not expose.

**What gets written to `generated.env`**, on top of a copy of the profile's `overrides.env`:

| Group | Variables |
| --- | --- |
| Paths and network | `VSS_APPS_DIR`, `VSS_DATA_DIR`, `HOST_IP`, `EXTERNAL_IP`, `VST_CONFIG_PATH` |
| Credentials | `NGC_CLI_API_KEY`, `NVIDIA_API_KEY`, `OPENAI_API_KEY` |
| Selection | `HARDWARE_PROFILE`, `MODE`, `INSTALL_PROPRIETARY_CODECS` |
| Brev secure links | `BREV_*`, `HAPROXY_PORT`, `VSS_PUBLIC_*` |
| LLM / VLM | `LLM_MODE`, `VLM_MODE`, `*_NAME`, `*_BASE_URL`, `*_DEVICE_ID`, `*_MODEL_TYPE`, `VLM_PORT` |
| Hardware tuning | `RTVI_VLLM_GPU_MEMORY_UTILIZATION`, `RTVI_VLM_MAX_MODEL_LEN`, `RT_VLM_DEVICE_ID`, `RTVI_VLM_MODEL_PATH` |
| Edge | `PERCEPTION_DOCKERFILE_PREFIX` |
| SBSA images | `VSS_RT_CV_TAG`, `VSS_RT_EMBED_TAG`, `VSS_RT_VLM_TAG`, `VSS_VIDEO_SUMMARIZATION_TAG` |

Supported hardware: `H100`, `L40S`, `RTXPRO4500BW`, `RTXPRO6000BW`, `DGX-SPARK`.

`DGX-SPARK` takes `base`, `alerts` and `search`, but not `lvs`. Its single GPU is already carrying the perception pipeline under `search`, so that profile requires **both** a remote LLM and a remote VLM; the cell then routes the agent straight at the VLM endpoint (`VLM_MODEL_TYPE=nim`, `VLM_AGENT_MEDIA_MODE=remote`), drops `rtvi-vlm` from `COMPOSE_PROFILES`, and collapses every placement onto device 0.

On a Brev Launchable, `search` with a local VLM needs at least 2 GPUs (RT-VLM sits next to RT-CV and RT-Embed); the cell counts GPUs and stops early, pointing at `USE_REMOTE_VLM` instead of failing during model startup.

Two more things happen on `DGX-SPARK`:

- **Cache cleaner.** Edge boards share one memory pool between CPU, GPU and the OS page cache, and without a periodic `drop_caches` the first inference frame OOMs. `prepare_deployment()` installs `/usr/local/bin/sys-cache-cleaner.sh` and starts it unless `pgrep` already finds it. It is deliberately not a systemd unit, so a reboot drops it and re-running the deploy cell restarts it.
- **`-sbsa` managed images.** The generic manifests carry the Tegra DeepStream build, which crash-loops on GB10's SBSA driver. `compose_env()` exports `VSS_CONTAINER_TAG_SUFFIX` for Compose interpolation, and because a service that reads a tag as plain configuration never sees that suffix, `generate_env()` also pins the four suffixed keys in `generated.env` (host env wins, then an uncommented env-file value, then the profile's commented `-sbsa` line, then the derived tag). Set `VSS_USE_SBSA_IMAGES=true` to force the same on other hardware.

In [ ]:
import os, re, json, glob, shutil, subprocess, time, urllib.request

DOCKER_DIR = os.path.join(REPO_DIR, "deploy", "docker")
DATA_DIR = os.path.join(DOCKER_DIR, "data-dir")
ALL_PROFILES = ("base", "lvs", "search", "alerts")
EDGE_HARDWARE_PROFILES = ("DGX-SPARK",)
# ARM64 hosts need the -sbsa managed image variants; the generic manifests carry
# the Tegra DeepStream build, which crash-loops on GB10's SBSA driver.
SBSA_HARDWARE_PROFILES = ("DGX-SPARK",)
SBSA_TAG_KEYS = ("VSS_RT_CV_TAG", "VSS_RT_EMBED_TAG", "VSS_RT_VLM_TAG",
                 "VSS_VIDEO_SUMMARIZATION_TAG")
MODE_ENV_VALUES = {"verification": "2d_cv", "real-time": "2d_vlm"}


def profile_path(profile, *parts):
    return os.path.join(DOCKER_DIR, "developer-profiles", f"dev-profile-{profile}", *parts)


def _host_env(name, default=""):
    return os.environ.get(name, default)


def _sudo_prefix():
    return ["sudo"] if os.geteuid() != 0 and shutil.which("sudo") else []


def mask_secret(secret):
    secret = str(secret)
    if len(secret) <= 6:
        return "******"
    return f"{secret[:3]}{'*' * (len(secret) - 6)}{secret[-3:]}"


# ============================================================
# env-file primitives
# ============================================================

def _strip_quotes(value):
    # One leading/trailing double quote, then one single quote — the quotes do
    # not have to pair up.
    if value.startswith('"'):
        value = value[1:]
    if value.endswith('"'):
        value = value[:-1]
    if value.startswith("'"):
        value = value[1:]
    if value.endswith("'"):
        value = value[:-1]
    return value


def read_env_value(path, name):
    if not os.path.isfile(path):
        return ""
    prefix = f"{name}="
    with open(path, encoding="utf-8") as f:
        for line in f:
            if line.startswith(prefix):
                return _strip_quotes(line.rstrip("\n").split("=", 1)[1])
    return ""


def env_file_has_var(path, name):
    if not os.path.isfile(path):
        return False
    prefix = f"{name}="
    with open(path, encoding="utf-8") as f:
        return any(line.startswith(prefix) for line in f)


def env_value_from_files(name, *paths):
    """Later files win over earlier ones."""
    value = ""
    for path in paths:
        if env_file_has_var(path, name):
            value = read_env_value(path, name)
    return value


def env_var_defined_in_files(name, *paths):
    return any(env_file_has_var(p, name) for p in paths)


def commented_sbsa_value(name, *paths):
    """Value of a commented SBSA alternate, e.g. `#VSS_RT_CV_TAG="develop-latest-sbsa"`.
    Profiles pin the SBSA build that way when it does not follow the shared
    VSS_CONTAINER_TAG channel. Later files win."""
    commented = re.compile(rf"^#[ \t]*{re.escape(name)}=")
    value = ""
    for path in paths:
        if not os.path.isfile(path):
            continue
        with open(path, encoding="utf-8") as f:
            for line in f:
                if commented.match(line) and "sbsa" in line:
                    value = _strip_quotes(line.rstrip("\n").split("=", 1)[1])
                    break
    return value


def use_sbsa_images(hardware_profile):
    return (hardware_profile in SBSA_HARDWARE_PROFILES
            or _host_env("VSS_USE_SBSA_IMAGES") == "true")


def set_env_var(path, name, value, mask=False):
    """Update an active assignment, else uncomment and update a commented one,
    else append. Profiles ship alternatives commented out, so the middle case
    is how a hardware or mode selection takes effect."""
    value = "" if value is None else str(value)
    with open(path, encoding="utf-8") as f:
        lines = f.readlines()

    active = re.compile(rf"^{re.escape(name)}=")
    commented = re.compile(rf"^#[ \t]*{re.escape(name)}=")
    replacement = f"{name}={value}\n"

    if any(active.match(l) for l in lines):
        lines = [replacement if active.match(l) else l for l in lines]
    elif any(commented.match(l) for l in lines):
        lines = [replacement if commented.match(l) else l for l in lines]
    else:
        lines.append(replacement)

    with open(path, "w", encoding="utf-8") as f:
        f.writelines(lines)
    print(f"[INFO] Set {name}={mask_secret(value) if mask else value}")


def _rewrite_matching(path, prefix, replacement, append_if_missing=False):
    """Replace every line starting with prefix. Returns True if anything changed."""
    with open(path, encoding="utf-8") as f:
        lines = f.readlines()
    matched = any(l.startswith(prefix) for l in lines)
    if matched:
        lines = [replacement if l.startswith(prefix) else l for l in lines]
    elif append_if_missing:
        lines.append(replacement)
    else:
        return False
    with open(path, "w", encoding="utf-8") as f:
        f.writelines(lines)
    return True


# ============================================================
# hardware detection and tuning
# ============================================================

def detect_gpu_names():
    try:
        out = subprocess.run(
            ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
            capture_output=True, text=True, timeout=30,
        )
    except (OSError, subprocess.SubprocessError):
        return []
    return [l.strip() for l in out.stdout.splitlines() if l.strip()]


def canonical_from_gpu_name(gpu_name):
    name = gpu_name.lower()
    if "h100" in name:
        return "H100"
    if "l40s" in name:
        return "L40S"
    if re.search(r"rtx.*pro.*4500.*blackwell", name):
        return "RTXPRO4500BW"
    if re.search(r"rtx.*pro.*6000.*blackwell", name):
        return "RTXPRO6000BW"
    if "gb10" in name:
        return "DGX-SPARK"
    return ""


def verify_hardware_profile(hardware_profile):
    if _host_env("SKIP_HARDWARE_CHECK").lower() == "true":
        return
    names = detect_gpu_names()
    if not names:
        raise RuntimeError(
            f"Hardware profile '{hardware_profile}' does not match detected hardware "
            "(no NVIDIA GPU detected). Set SKIP_HARDWARE_CHECK=true to bypass."
        )
    # Mixed-GPU hosts must not be rejected because a different card is listed first.
    if not any(canonical_from_gpu_name(n) == hardware_profile for n in names):
        raise RuntimeError(
            f"Hardware profile '{hardware_profile}' does not match any detected NVIDIA "
            f"GPU: {', '.join(names)}. Set SKIP_HARDWARE_CHECK=true to bypass."
        )
    print(f"[INFO] Hardware profile {hardware_profile} matches: {', '.join(names)}")


def rtvi_vllm_gpu_memory_utilization(hardware_profile, vlm_mode, profile):
    # Edge boards draw CPU, GPU and page cache from one pool, so RT-VLM takes a
    # smaller slice than on a dGPU. alerts needs it whether or not the LLM is
    # co-resident, because the CV pipeline is on the same GPU either way.
    if hardware_profile == "DGX-SPARK" and (profile == "alerts" or vlm_mode == "local_shared"):
        return "0.35"
    if hardware_profile in ("L40S", "RTXPRO4500BW"):
        return "0.8"
    # vLLM claims the whole fraction whether it needs it or not and refuses to
    # start unless free >= fraction x total, so a co-located LLM needs the rest
    # of the card left free.
    return "0.4" if vlm_mode == "local_shared" else "0.7"


CACHE_CLEANER_PATH = "/usr/local/bin/sys-cache-cleaner.sh"
CACHE_CLEANER_SCRIPT = """#!/bin/bash
set -e
echo 0 | tee /proc/sys/vm/nr_hugepages
echo "Starting cache cleaner"
while true; do
  sync && echo 3 | tee /proc/sys/vm/drop_caches > /dev/null
  sleep 3
done
"""


def cache_cleaner_running():
    """None when pgrep is missing, so the caller can skip the check instead of
    treating an unanswerable question as a failure."""
    try:
        return subprocess.run(["pgrep", "-f", CACHE_CLEANER_PATH],
                              capture_output=True).returncode == 0
    except OSError:
        return None


def ensure_cache_cleaner(hardware_profile):
    """Edge boards share one memory pool between CPU, GPU and the OS page cache.
    Without a periodic drop_caches the cache pins enough of it that the first
    inference frame OOMs, so the prerequisites list the cleaner as a platform
    requirement. It is deliberately not a systemd unit, so a reboot drops it and
    every deployment has to check again."""
    if hardware_profile not in EDGE_HARDWARE_PROFILES:
        return
    if cache_cleaner_running():
        print(f"[INFO] Cache cleaner already running ({CACHE_CLEANER_PATH})")
        return

    print(f"[INFO] {hardware_profile}: installing and starting the cache cleaner...")
    sudo = _sudo_prefix()
    subprocess.run(sudo + ["tee", CACHE_CLEANER_PATH], input=CACHE_CLEANER_SCRIPT,
                   text=True, check=True, stdout=subprocess.DEVNULL)
    subprocess.run(sudo + ["chmod", "+x", CACHE_CLEANER_PATH], check=True)
    subprocess.Popen(sudo + [CACHE_CLEANER_PATH], stdout=subprocess.DEVNULL,
                     stderr=subprocess.DEVNULL, start_new_session=True)
    time.sleep(2)
    if cache_cleaner_running() is False:
        raise RuntimeError(
            f"Cache cleaner did not start. Run it manually before deploying: "
            f"sudo -b {CACHE_CLEANER_PATH}"
        )
    print("[INFO] Cache cleaner started (stops on reboot; re-run the deploy cell "
          "to restart it)")


def rtvi_vlm_max_model_len(hardware_profile):
    return "18000" if hardware_profile == "RTXPRO4500BW" else ""


# ============================================================
# LLM / VLM mode derivation
# ============================================================

def remote_model_name(base_url, expected_type):
    """Auto-detect the served model. Only safe when the endpoint serves exactly
    one model: aggregate endpoints list every NIM and the first entry is not a
    meaningful default."""
    url = f"{base_url}/v1/models"
    try:
        with urllib.request.urlopen(url, timeout=15) as resp:
            body = json.loads(resp.read().decode())
    except Exception as exc:
        raise RuntimeError(f"Failed to retrieve model list from {url}: {exc}") from None

    models = [m.get("id") for m in body.get("data", []) if m.get("id")]
    if not models:
        raise RuntimeError(f"No models returned from {url}")
    if len(models) > 1:
        listing = "\n".join(f"    {m}" for m in models)
        raise RuntimeError(
            f"{url} returns {len(models)} models — auto-select is unsafe. Point "
            f"REMOTE_{expected_type.upper()}_ENDPOINT_URL at a single-model "
            f"endpoint. Available:\n{listing}"
        )
    return models[0]


def derive_modes(profile, hardware_profile, use_remote_llm, use_remote_vlm,
                 llm_endpoint_url, vlm_endpoint_url):
    source_env = profile_path(profile, ".env")
    overrides_env = profile_path(profile, "overrides.env")

    llm_base_url = llm_endpoint_url if use_remote_llm else ""
    vlm_base_url = vlm_endpoint_url if use_remote_vlm else ""
    if use_remote_llm and not llm_base_url:
        raise RuntimeError("REMOTE_LLM_ENDPOINT_URL must be set when USE_REMOTE_LLM=True")
    if use_remote_vlm and not vlm_base_url:
        raise RuntimeError("REMOTE_VLM_ENDPOINT_URL must be set when USE_REMOTE_VLM=True")
    llm_is_remote = bool(llm_base_url)
    vlm_is_remote = bool(vlm_base_url)

    if hardware_profile in EDGE_HARDWARE_PROFILES:
        # Edge platforms are single-GPU; device IDs are not configurable.
        llm_device_id = vlm_device_id = "0"
    else:
        llm_device_id = env_value_from_files("LLM_DEVICE_ID", source_env, overrides_env)
        vlm_device_id = env_value_from_files("VLM_DEVICE_ID", source_env, overrides_env)

    def _id_list(name):
        raw = env_value_from_files(name, source_env, overrides_env).replace(" ", "")
        return f",{raw},"

    fixed_shared = _id_list("FIXED_SHARED_DEVICE_IDS")
    reserved = _id_list("RESERVED_DEVICE_IDS")

    def _mode(is_remote, own_id, other_id, other_is_remote):
        if is_remote:
            return "remote"
        if not own_id:
            return "local"
        if f",{own_id}," in reserved or f",{own_id}," in fixed_shared:
            return "local_shared"
        if not other_is_remote and own_id == other_id:
            return "local_shared"
        return "local"

    llm_mode = _mode(llm_is_remote, llm_device_id, vlm_device_id, vlm_is_remote)
    vlm_mode = _mode(vlm_is_remote, vlm_device_id, llm_device_id, llm_is_remote)

    # L40S (48 GB) has no shared-LLM profile.
    if hardware_profile == "L40S":
        if llm_mode == "local_shared":
            raise RuntimeError("On L40S, the device ID for the LLM cannot be shared with other services")
        if (llm_mode != "remote" and vlm_mode != "remote"
                and llm_device_id and llm_device_id == vlm_device_id):
            raise RuntimeError("On L40S, the LLM and VLM cannot share the same GPU")

    if hardware_profile not in EDGE_HARDWARE_PROFILES:
        for label, mode, dev in (("LLM", llm_mode, llm_device_id), ("VLM", vlm_mode, vlm_device_id)):
            if mode != "remote" and dev and f",{dev}," in reserved:
                raise RuntimeError(
                    f"Device ID {dev} is reserved and cannot be assigned to {label} for this profile"
                )

    return {
        "llm_mode": llm_mode, "vlm_mode": vlm_mode,
        "llm_device_id": llm_device_id, "vlm_device_id": vlm_device_id,
        "llm_base_url": llm_base_url, "vlm_base_url": vlm_base_url,
    }


# ============================================================
# alerts MODE derivatives
# ============================================================

def apply_alerts_mode(generated_env):
    mode = read_env_value(generated_env, "MODE")
    if mode not in ("2d_cv", "2d_vlm"):
        return

    subtitle = "Vision (Alerts - CV)" if mode == "2d_cv" else "Vision (Alerts - VLM)"
    if _rewrite_matching(generated_env, "NEXT_PUBLIC_APP_SUBTITLE=",
                         f'NEXT_PUBLIC_APP_SUBTITLE="{subtitle}"\n'):
        print(f"[INFO] Set NEXT_PUBLIC_APP_SUBTITLE for alerts (MODE={mode} → {subtitle})")

    # Verification keeps both Manage Alerts editors; real-time drops the CV one.
    realtime, verification = "true", ("true" if mode == "2d_cv" else "false")
    for name, value in (
        ("NEXT_PUBLIC_ALERTS_TAB_MANAGE_ALERTS_SUB_TAB_ENABLE_REALTIME_ALERTS", realtime),
        ("NEXT_PUBLIC_ALERTS_TAB_MANAGE_ALERTS_SUB_TAB_ENABLE_CV_ALERTS_VERIFICATION", verification),
    ):
        _rewrite_matching(generated_env, f"{name}=", f"{name}={value}\n", append_if_missing=True)
    print(f"[INFO] Set Alerts Manage tabs for MODE={mode} "
          f"(real-time={realtime}, CV verification={verification})")

    # Real-time drives alerts from RT-VLM's Kafka events, so drop the
    # overrides.env value and let the rtvi-vlm compose default (true) apply.
    if mode == "2d_vlm":
        with open(generated_env, encoding="utf-8") as f:
            lines = f.readlines()
        if any(l.startswith("RTVI_VLM_KAFKA_ENABLED=") for l in lines):
            with open(generated_env, "w", encoding="utf-8") as f:
                f.writelines("#" + l if l.startswith("RTVI_VLM_KAFKA_ENABLED=") else l
                             for l in lines)
            print("[INFO] Commented out RTVI_VLM_KAFKA_ENABLED for alerts (MODE=2d_vlm)")


# ============================================================
# generated.env
# ============================================================

def generate_env(profile, hardware_profile, host_ip, external_ip, modes, mode_env, ngc_api_key):
    source_env = profile_path(profile, ".env")
    overrides_env = profile_path(profile, "overrides.env")
    generated_env = profile_path(profile, "generated.env")

    print(f"[INFO] Generating environment file for profile '{profile}'...")
    if not os.path.isfile(source_env):
        raise FileNotFoundError(f"Source .env file not found: {source_env}")
    if not os.path.isfile(overrides_env):
        raise FileNotFoundError(f"Overrides env file not found: {overrides_env}")

    shutil.copyfile(overrides_env, generated_env)
    print(f"[INFO] Copied {overrides_env} to {generated_env}")

    # A missing final newline would merge the first appended variable into the
    # last existing one.
    with open(generated_env, "rb") as f:
        content = f.read()
    if content and not content.endswith(b"\n"):
        with open(generated_env, "ab") as f:
            f.write(b"\n")

    # Compose-wide defaults, for variables the profile does not define.
    compose_defaults = os.path.join(DOCKER_DIR, "services", "vios", "compose-defaults.env")
    if os.path.isfile(compose_defaults):
        with open(compose_defaults, encoding="utf-8") as f:
            default_lines = f.readlines()
        for line in default_lines:
            entry = line.rstrip("\n")
            if not entry.strip() or entry.lstrip().startswith("#"):
                continue
            if not env_var_defined_in_files(entry.split("=", 1)[0], source_env, generated_env):
                with open(generated_env, "a", encoding="utf-8") as out:
                    out.write(entry + "\n")

    set_env_var(generated_env, "VSS_APPS_DIR", DOCKER_DIR)
    set_env_var(generated_env, "VSS_DATA_DIR", DATA_DIR)
    set_env_var(generated_env, "HOST_IP", host_ip)
    # Forwarded so the agent/RTVI services install the patent-encumbered codecs
    # at startup. Export INSTALL_PROPRIETARY_CODECS=false to keep them off.
    set_env_var(generated_env, "INSTALL_PROPRIETARY_CODECS",
                _host_env("INSTALL_PROPRIETARY_CODECS", "true"))
    set_env_var(generated_env, "VST_CONFIG_PATH",
                os.path.join(DOCKER_DIR, "services", "vios", "configs"))
    if external_ip:
        set_env_var(generated_env, "EXTERNAL_IP", external_ip, mask=True)

    # ===== Brev secure links =====
    brev_env_id = _host_env("BREV_ENV_ID")
    if brev_env_id:
        proxy_port = _host_env("PROXY_PORT", "7777")
        link_prefix = _host_env("BREV_LINK_PREFIX", proxy_port)
        link_domain = _host_env("BREV_LINK_DOMAIN") or detect_brev_link_domain()
        secure_link_host = f"{link_prefix}-{brev_env_id}.{link_domain}"
        print(f"[INFO] Brev environment detected ({brev_env_id}). "
              f"Setting HAProxy ingress to {secure_link_host}...")
        for name, value in (
            ("BREV_ENV_ID", brev_env_id),
            ("BREV_LINK_PREFIX", link_prefix),
            ("BREV_LINK_DOMAIN", link_domain),
            ("HAPROXY_PORT", proxy_port),
            ("VSS_PUBLIC_HTTP_PROTOCOL", "https"),
            ("VSS_PUBLIC_WS_PROTOCOL", "wss"),
            ("VSS_PUBLIC_HOST", secure_link_host),
            ("VSS_PUBLIC_PORT", "443"),
        ):
            set_env_var(generated_env, name, value)

    set_env_var(generated_env, "NGC_CLI_API_KEY", ngc_api_key, mask=True)
    set_env_var(generated_env, "HARDWARE_PROFILE", hardware_profile)
    if mode_env:
        set_env_var(generated_env, "MODE", mode_env)

    if profile == "alerts":
        apply_alerts_mode(generated_env)
        if read_env_value(generated_env, "MODE") == "2d_vlm":
            set_env_var(generated_env, "COMPOSE_PROFILES", "${COMPOSE_PROFILES_VLM}")
            set_env_var(generated_env, "ALERT_AGENT_ALWAYS_ON", "true")
        else:
            set_env_var(generated_env, "ALERT_AGENT_ALWAYS_ON", "false")

    llm_mode, vlm_mode = modes["llm_mode"], modes["vlm_mode"]
    llm_base_url, vlm_base_url = modes["llm_base_url"], modes["vlm_base_url"]

    # ===== LLM =====
    set_env_var(generated_env, "LLM_MODE", llm_mode)
    if llm_mode == "remote":
        set_env_var(generated_env, "LLM_NAME", remote_model_name(llm_base_url, "llm"))
        set_env_var(generated_env, "LLM_NAME_SLUG", "none")
    if hardware_profile in EDGE_HARDWARE_PROFILES:
        set_env_var(generated_env, "LLM_DEVICE_ID", "0")
        set_env_var(generated_env, "VLM_DEVICE_ID", "0")
        # Search's profile .env pins a 2-GPU layout (RT-CV on 0, RT-Embed on 1).
        # These boards have one GPU, and a device_ids entry of "1" is a hard
        # startup failure, so collapse every placement onto device 0.
        if profile == "search":
            set_env_var(generated_env, "RT_CV_DEVICE_ID", "0")
            set_env_var(generated_env, "RT_EMBED_DEVICE_ID", "0")
            set_env_var(generated_env, "FIXED_SHARED_DEVICE_IDS", "0")
    else:
        if llm_mode != "remote" and modes["llm_device_id"]:
            set_env_var(generated_env, "LLM_DEVICE_ID", modes["llm_device_id"])
        if vlm_mode != "remote" and modes["vlm_device_id"]:
            set_env_var(generated_env, "VLM_DEVICE_ID", modes["vlm_device_id"])
    if llm_base_url:
        set_env_var(generated_env, "LLM_BASE_URL", llm_base_url)
    if llm_mode == "remote":
        llm_model_type = env_value_from_files("LLM_MODEL_TYPE", source_env, overrides_env)
        if llm_model_type:
            set_env_var(generated_env, "LLM_MODEL_TYPE", llm_model_type)
    if _host_env("NVIDIA_API_KEY"):
        set_env_var(generated_env, "NVIDIA_API_KEY", _host_env("NVIDIA_API_KEY"), mask=True)

    # ===== VLM =====
    set_env_var(generated_env, "VLM_MODE", vlm_mode)
    if vlm_mode == "remote":
        set_env_var(generated_env, "VLM_NAME", remote_model_name(vlm_base_url, "vlm"))
        set_env_var(generated_env, "VLM_NAME_SLUG", "none")
        set_env_var(generated_env, "VLM_BASE_URL", vlm_base_url)
        set_env_var(generated_env, "RTVI_VLM_ENDPOINT", f"{vlm_base_url}/v1")
        set_env_var(generated_env, "RTVI_VLM_MODEL_PATH", "none")
        vlm_model_type = env_value_from_files("VLM_MODEL_TYPE", source_env, overrides_env)
        if vlm_model_type:
            set_env_var(generated_env, "VLM_MODEL_TYPE", vlm_model_type)
    if _host_env("OPENAI_API_KEY"):
        set_env_var(generated_env, "OPENAI_API_KEY", _host_env("OPENAI_API_KEY"), mask=True)

    if vlm_mode == "remote":
        # rtvi-vlm proxies to the endpoint, so use the conventional NIM port and
        # openai-compat instead of the integrated local checkpoint.
        set_env_var(generated_env, "VLM_PORT", "30082")
        set_env_var(generated_env, "RTVI_VLM_MODEL_TO_USE", "openai-compat")
        if profile == "lvs":
            # LVS defers frame sampling to RT-VLM.
            configured = env_value_from_files(
                "RTVI_VLM_DEFAULT_NUM_FRAMES_PER_SECOND_OR_FIXED_FRAMES_CHUNK",
                source_env, overrides_env)
            frames = _host_env(
                "RTVI_VLM_DEFAULT_NUM_FRAMES_PER_SECOND_OR_FIXED_FRAMES_CHUNK", configured)
            set_env_var(generated_env,
                        "RTVI_VLM_DEFAULT_NUM_FRAMES_PER_SECOND_OR_FIXED_FRAMES_CHUNK",
                        frames or "5")
    else:
        custom_weights = _host_env("VLM_CUSTOM_WEIGHTS") or env_value_from_files(
            "VLM_CUSTOM_WEIGHTS", source_env, overrides_env)
        if custom_weights:
            print(f"[INFO] Using VLM custom weights path: {custom_weights}")
            set_env_var(generated_env, "VLM_CUSTOM_WEIGHTS", custom_weights)

    # ===== Edge and hardware tuning =====
    if profile == "alerts" and hardware_profile in EDGE_HARDWARE_PROFILES:
        set_env_var(generated_env, "PERCEPTION_DOCKERFILE_PREFIX", "EDGE-")

    set_env_var(generated_env, "VLM_NAME_SLUG", "none")
    if vlm_mode != "remote":
        # Internal URL so sibling containers need no host-published ports.
        set_env_var(generated_env, "VLM_BASE_URL", "http://rtvi-vlm:8000")
        set_env_var(generated_env, "RTVI_VLLM_GPU_MEMORY_UTILIZATION",
                    rtvi_vllm_gpu_memory_utilization(hardware_profile, vlm_mode, profile))
        max_model_len = rtvi_vlm_max_model_len(hardware_profile)
        if max_model_len:
            set_env_var(generated_env, "RTVI_VLM_MAX_MODEL_LEN", max_model_len)

    if vlm_mode == "local_shared":
        shared_id = env_value_from_files("SHARED_LLM_VLM_DEVICE_ID", source_env, overrides_env)
        set_env_var(generated_env, "RT_VLM_DEVICE_ID", shared_id or modes["vlm_device_id"])
    elif vlm_mode == "remote":
        set_env_var(generated_env, "RT_VLM_DEVICE_ID", "0")
    else:
        set_env_var(generated_env, "RT_VLM_DEVICE_ID", modes["vlm_device_id"])

    if hardware_profile == "RTXPRO4500BW" and vlm_mode != "remote":
        set_env_var(generated_env, "RTVI_VLM_MODEL_PATH",
                    "ngc:nim/nvidia/cosmos3-nano-reasoner:bf16-final")
        set_env_var(generated_env, "VLM_NAME", "nim_nvidia_cosmos3-nano-reasoner_bf16-final")

    # Search on edge hardware: the VLM is always a remote endpoint, and the agent
    # calls it directly instead of proxying through RT-VLM, which frees the GPU
    # that container would otherwise hold for media preprocessing.
    if profile == "search" and hardware_profile in EDGE_HARDWARE_PROFILES:
        set_env_var(generated_env, "VLM_MODEL_TYPE", "nim")
        set_env_var(generated_env, "VLM_AGENT_MEDIA_MODE", "remote")
        compose_profiles = read_env_value(generated_env, "COMPOSE_PROFILES")
        kept = [p for p in compose_profiles.split(",") if p != "rtvi-vlm"]
        if len(kept) != len(compose_profiles.split(",")):
            set_env_var(generated_env, "COMPOSE_PROFILES", ",".join(kept))
            print("[INFO] Removed rtvi-vlm from COMPOSE_PROFILES (search on "
                  f"{hardware_profile} calls the remote VLM directly)")

    if profile == "base" and vlm_mode != "remote":
        set_env_var(generated_env, "VLM_MODEL_TYPE", "rtvi")
        compose_profiles = read_env_value(generated_env, "COMPOSE_PROFILES")
        if "rtvi-vlm" not in compose_profiles.split(","):
            set_env_var(generated_env, "COMPOSE_PROFILES", f"{compose_profiles},rtvi-vlm")

    # ===== SBSA managed images =====
    # containers.env applies VSS_CONTAINER_TAG_SUFFIX during Compose
    # interpolation only, so a service that reads a tag as plain configuration
    # never sees it. Write the four suffixed keys here too, preferring an
    # explicit tag (host env, then an uncommented env-file line, then the
    # profile's commented SBSA pin) over the derived one.
    if use_sbsa_images(hardware_profile):
        base_tag = (_host_env("VSS_CONTAINER_TAG")
                    or env_value_from_files("VSS_CONTAINER_TAG", source_env, generated_env))
        for key in SBSA_TAG_KEYS:
            value = _host_env(key) or env_value_from_files(key, source_env, generated_env)
            if not value and not base_tag:
                # No shared channel selected, so the profile's pinned SBSA build
                # is the only meaningful tag. A selected channel always wins.
                value = commented_sbsa_value(key, source_env, generated_env)
            set_env_var(generated_env, key, value or f"{base_tag or 'develop-latest'}-sbsa")

    print(f"[INFO] Generated environment file: {generated_env}")
    return generated_env


# ============================================================
# directories, kernel settings, Compose
# ============================================================

def create_directories(profile):
    print("[INFO] Creating data directories...")
    for rel in ("data_log/analytics_cache", "data_log/calibration_toolkit",
                "data_log/elastic/data", "data_log/elastic/logs", "data_log/kafka",
                "data_log/redis/data", "data_log/redis/log",
                "agent_eval/dataset", "agent_eval/results"):
        os.makedirs(os.path.join(DATA_DIR, rel), exist_ok=True)

    if profile == "alerts":
        print("[INFO] Creating alerts-specific directories...")
        os.makedirs(os.path.join(DATA_DIR, "data_log", "vss_video_analytics_api"), exist_ok=True)
        os.makedirs(os.path.join(DATA_DIR, "videos", "dev-profile-alerts"), exist_ok=True)
        engines = os.path.join(DOCKER_DIR, "engines")
        os.makedirs(os.path.join(engines, "gdino"), exist_ok=True)
        os.makedirs(os.path.join(engines, "rtdetr-its"), exist_ok=True)
        subprocess.run(["chmod", "-R", "777", engines], check=False)
        print("[INFO] Alerts model download runs in ds-start.sh phase 0 (perception).")

    if profile == "search":
        print("[INFO] Creating search-specific directories...")
        os.makedirs(os.path.join(DATA_DIR, "data_log", "vss_video_analytics_api"), exist_ok=True)
        print("[INFO] Search model download runs in ds-start.sh phase 0 (perception).")

    for rel in ("data_log", "agent_eval"):
        print(f"[INFO] Setting permissions on {rel} directory...")
        subprocess.run(["chmod", "-R", "777", os.path.join(DATA_DIR, rel)], check=False)


def apply_kernel_settings():
    """IPv6 off plus larger TCP buffers, persisted across reboots."""
    print("[INFO] Applying VSS Linux kernel settings...")
    settings = "".join(f"{line}\n" for line in (
        "net.ipv6.conf.all.disable_ipv6 = 1",
        "net.ipv6.conf.default.disable_ipv6 = 1",
        "net.ipv6.conf.lo.disable_ipv6 = 1",
        "net.core.rmem_max = 5242880",
        "net.core.wmem_max = 5242880",
        "net.ipv4.tcp_rmem = 4096 87380 16777216",
        "net.ipv4.tcp_wmem = 4096 65536 16777216",
    ))
    sudo = _sudo_prefix()
    subprocess.run(sudo + ["mkdir", "-p", "/etc/sysctl.d"], check=True)
    subprocess.run(sudo + ["bash", "-c", "cat > /etc/sysctl.d/99-vss.conf"],
                   input=settings, text=True, check=True)
    subprocess.run(sudo + ["sysctl", "--system"], capture_output=True, check=False)


def compose_base_args(profile):
    """Compose invocation shared by up, stop and image resolution. Paths are
    relative to deploy/docker, so callers must run from DOCKER_DIR."""
    # -f disables Compose's default file discovery, so name the base file even
    # when there is no overlay.
    args = ["docker", "compose", "-f", "compose.yml"]
    if _host_env("VSS_VIOS_PREBAKE_PACKAGES", "false") == "true":
        args += ["-f", "services/vios/streamprocessing/docker-compose.prebaked.yaml"]
    args += [
        "--env-file", "containers.env",
        "--env-file", f"developer-profiles/dev-profile-{profile}/.env",
        "--env-file", f"developer-profiles/dev-profile-{profile}/generated.env",
    ]
    return args


def compose_env(hardware_profile):
    """Process environment for every Compose call.

    Compose gives the process environment precedence over all --env-file
    arguments, so the SBSA image suffix and the managed registry/tag pair have
    to be set here — writing them to generated.env would be silently overridden
    by containers.env. Per-image overrides in generated.env still apply, which
    is where generate_env() pins the four suffixed service tags.
    """
    env = dict(os.environ)

    if use_sbsa_images(hardware_profile):
        env["VSS_CONTAINER_TAG_SUFFIX"] = "-sbsa"

    # containers.env defines its values with shell parameter expansion, so let
    # bash evaluate it rather than parsing the lines.
    resolve = ('set -a; . "$1"; set +a; '
               'printf "%s\\n%s\\n" "${VSS_CONTAINER_REGISTRY}" "${VSS_CONTAINER_TAG}"')
    result = subprocess.run(
        ["bash", "-c", resolve, "bash", os.path.join(DOCKER_DIR, "containers.env")],
        capture_output=True, text=True, env=env)
    if result.returncode != 0:
        raise RuntimeError(f"Could not resolve containers.env:\n{result.stderr}")
    registry, tag = (result.stdout.splitlines() + ["", ""])[:2]
    env["VSS_CONTAINER_REGISTRY"] = registry
    env["VSS_CONTAINER_TAG"] = tag
    return env


def show_resolved_images(profile, hardware_profile):
    env = compose_env(hardware_profile)
    if env.get("VSS_CONTAINER_TAG_SUFFIX"):
        print(f"[INFO] Managed container tag suffix: {env['VSS_CONTAINER_TAG_SUFFIX']}")
    print(f"[INFO] Managed container registry: {env['VSS_CONTAINER_REGISTRY']}")
    print(f"[INFO] Managed container tag:      {env['VSS_CONTAINER_TAG']}")
    print("[INFO] Resolved compose images:")
    result = subprocess.run(compose_base_args(profile) + ["config", "--images"],
                            cwd=DOCKER_DIR, capture_output=True, text=True, env=env)
    if result.returncode != 0:
        raise RuntimeError(f"docker compose config failed:\n{result.stderr}")
    for image in sorted({l.strip() for l in result.stdout.splitlines() if l.strip()}):
        print(f"  {image}")


def docker_login_nvcr(ngc_api_key):
    print("[INFO] Logging into nvcr.io...")
    result = subprocess.run(
        ["docker", "login", "--username", "$oauthtoken", "--password", ngc_api_key, "nvcr.io"],
        capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"docker login to nvcr.io failed:\n{result.stderr}")


# ============================================================
# teardown
# ============================================================

def compose_project_names():
    """Every project name that might have a running deployment."""
    names = []
    if _host_env("COMPOSE_PROJECT_NAME"):
        names.append(_host_env("COMPOSE_PROJECT_NAME"))
    for prof in ALL_PROFILES:
        generated = profile_path(prof, "generated.env")
        if os.path.isfile(generated):
            name = env_value_from_files(
                "COMPOSE_PROJECT_NAME",
                profile_path(prof, ".env"), profile_path(prof, "overrides.env"), generated,
            ) or "vss"
            if name not in names:
                names.append(name)
    if not names:
        for prof in ALL_PROFILES:
            name = env_value_from_files(
                "COMPOSE_PROJECT_NAME",
                profile_path(prof, ".env"), profile_path(prof, "overrides.env"))
            if name:
                names.append(name)
                break
    return names or ["vss"]


def teardown_deployment():
    for name in compose_project_names():
        print(f"[INFO] Bringing down docker compose project '{name}' (with volumes)...")
        result = subprocess.run(["docker", "compose", "-p", name, "down", "-v", "--remove-orphans"],
                                cwd=SCRIPT_DIR, capture_output=True, text=True)
        for line in (result.stdout + result.stderr).splitlines():
            if line.strip() and "is not set" not in line:
                print(f"  {line}")

    print("[INFO] Cleaning up generated.env files from all profiles...")
    for prof in ALL_PROFILES:
        generated = profile_path(prof, "generated.env")
        if os.path.isfile(generated):
            os.remove(generated)
            print(f"[INFO] Deleted {generated}")

    print("[INFO] Removing dangling docker volumes...")
    dangling = subprocess.run(["docker", "volume", "ls", "-q", "-f", "dangling=true"],
                              capture_output=True, text=True).stdout.split()
    if dangling:
        subprocess.run(["docker", "volume", "rm"] + dangling, capture_output=True, text=True)
        print(f"[INFO] Removed {len(dangling)} dangling volume(s)")
    else:
        print("[INFO] No dangling volumes to remove")

    sudo = _sudo_prefix()

    # sdrc artifacts are bind-mounted and written by containers as root.
    sdrc_dir = os.path.join(DOCKER_DIR, "services", "infra", "sdrc")
    print(f"[INFO] Cleaning up sdrc runtime artifacts in {sdrc_dir}...")
    for artifact in (os.path.join(sdrc_dir, "log"), os.path.join(sdrc_dir, ".wdm-env")):
        if os.path.isdir(artifact):
            subprocess.run(sudo + ["rm", "-rf", artifact], check=False)
            print(f"[INFO] Deleted {artifact}")

    # Every rendered sdrc config has a *.tmpl sibling; drop the rendered copy so
    # the next run regenerates it from the template.
    seen = set()
    for pattern in (os.path.join(DOCKER_DIR, "**", "sdrc", "configs", "*.tmpl"),
                    os.path.join(DOCKER_DIR, "**", "sdrc", "*", "configs", "*.tmpl")):
        for tmpl in glob.glob(pattern, recursive=True):
            rendered = tmpl[: -len(".tmpl")]
            if rendered in seen or not os.path.isfile(rendered):
                continue
            seen.add(rendered)
            subprocess.run(sudo + ["rm", "-f", rendered], check=False)
            print(f"[INFO] Deleted rendered sdrc config: {rendered}")

    print(f"[INFO] Deleting data directory: {DATA_DIR}...")
    if os.path.isdir(DATA_DIR):
        subprocess.run(sudo + ["rm", "-rf", DATA_DIR], check=False)
        print("[INFO] Data directory deleted")
    else:
        print("[INFO] Data directory does not exist, skipping")

    print("[INFO] State down completed")


# ============================================================
# orchestration
# ============================================================

def prepare_deployment(profile, hardware_profile, host_ip, external_ip,
                       use_remote_llm, use_remote_vlm,
                       llm_endpoint_url, vlm_endpoint_url, alerts_mode, ngc_api_key):
    """Everything that has to happen before `docker compose up`."""
    if hardware_profile in EDGE_HARDWARE_PROFILES:
        if profile not in ("base", "alerts", "search"):
            raise RuntimeError(
                f"Hardware profile '{hardware_profile}' is only valid for profile base, "
                f"alerts or search, not '{profile}'"
            )
        # One GPU, and search's perception pipeline already owns it: a local VLM
        # never fits, and two vLLM engines cannot share a single GPU at all.
        if profile == "search":
            if not (use_remote_llm and llm_endpoint_url):
                raise RuntimeError(
                    f"Search on {hardware_profile} cannot host a local LLM. Set "
                    "USE_REMOTE_LLM=True with REMOTE_LLM_ENDPOINT_URL."
                )
            if not (use_remote_vlm and vlm_endpoint_url):
                raise RuntimeError(
                    f"Search on {hardware_profile} cannot host a local VLM. Set "
                    "USE_REMOTE_VLM=True with REMOTE_VLM_ENDPOINT_URL."
                )

    verify_hardware_profile(hardware_profile)
    ensure_cache_cleaner(hardware_profile)

    mode_env = MODE_ENV_VALUES.get(alerts_mode, "") if profile == "alerts" else ""
    modes = derive_modes(profile, hardware_profile, use_remote_llm, use_remote_vlm,
                         llm_endpoint_url, vlm_endpoint_url)
    print(f"[INFO] LLM mode: {modes['llm_mode']}    VLM mode: {modes['vlm_mode']}")

    # Search brings up a local RT-VLM next to RT-CV and RT-Embed, so the profile
    # layout needs two GPUs. Without this the run fails much later, during
    # Compose or model startup.
    if profile == "search" and _host_env("BREV_ENV_ID") and modes["vlm_mode"] != "remote":
        gpu_count = len(detect_gpu_names())
        if 0 < gpu_count < 2:
            raise RuntimeError(
                f"Search deploys a local RT-VLM and needs at least 2 GPUs, but this Brev "
                f"environment has {gpu_count} GPU(s). Set USE_REMOTE_VLM=True with "
                "REMOTE_VLM_ENDPOINT_URL to run search here."
            )

    # Deploying always starts from a clean slate.
    teardown_deployment()

    generate_env(profile, hardware_profile, host_ip, external_ip, modes, mode_env, ngc_api_key)
    create_directories(profile)
    apply_kernel_settings()
    show_resolved_images(profile, hardware_profile)
    docker_login_nvcr(ngc_api_key)
    return modes


print("Deployment helpers loaded.")
print(f"  Deployment directory: {DOCKER_DIR}")
print(f"  Data directory:       {DATA_DIR}")

## 8. Deploy Profile

This is the main deployment cell. It uses the helpers from Section 7.1 with the configuration from Section 1 and the network settings from Section 7, then calls `docker compose up` directly.

This will:
- **Tear down any existing deployment first**, including volumes and the data directory
- Generate `generated.env` for the selected profile and hardware
- Create the data directories and apply the VSS kernel settings
- Download required models from NGC
- Pull and build Docker images
- Start all containers

> **Deploying wipes existing data.** Every run starts from a clean slate, so uploaded videos, Elasticsearch indices and Kafka data from a previous deployment are removed. Use Section 13 to stop containers without losing data.

**This cell takes 10-30 minutes** depending on network speed and whether images are cached.

The cell shows a live progress summary. Full output is captured to `~/deploy_vss.log` — if something fails, check that file for details.

In [ ]:
import subprocess, os, re, time, datetime, io, contextlib
from IPython.display import display, clear_output, HTML

LOG_FILE = os.path.expanduser("~/deploy_vss.log")

# Validate the hardware, tear down any previous deployment, write generated.env,
# create data directories, apply kernel settings and log into nvcr.io.
# This output is verbose and the live status display below clears the cell, so
# capture it into the log file and surface a one-line summary.
prep_buffer = io.StringIO()
try:
    with contextlib.redirect_stdout(prep_buffer):
        prepare_deployment(
            profile=PROFILE,
            hardware_profile=HARDWARE_PROFILE,
            host_ip=HOST_IP,
            external_ip=EXTERNAL_IP if (EXTERNAL_IP and EXTERNAL_IP != HOST_IP) else "",
            use_remote_llm=USE_REMOTE_LLM,
            use_remote_vlm=USE_REMOTE_VLM,
            llm_endpoint_url=REMOTE_LLM_ENDPOINT_URL,
            vlm_endpoint_url=REMOTE_VLM_ENDPOINT_URL,
            alerts_mode=ALERTS_MODE,
            ngc_api_key=NGC_CLI_API_KEY,
        )
finally:
    with open(LOG_FILE, "w") as log:
        log.write(prep_buffer.getvalue())

_vars_set = sum(1 for l in prep_buffer.getvalue().splitlines() if l.startswith("[INFO] Set "))
print(f"Environment prepared: {_vars_set} variables written to "
      f"developer-profiles/dev-profile-{PROFILE}/generated.env")

cmd = compose_base_args(PROFILE) + [
    "up", "--detach", "--pull", "always", "--force-recreate", "--build",
]

# compose_env() carries the SBSA image suffix and the managed registry/tag pair,
# which Compose only honours from the process environment.
deploy_env = {**compose_env(HARDWARE_PROFILE), "NGC_CLI_API_KEY": NGC_CLI_API_KEY}
if USE_REMOTE_LLM:
    deploy_env["LLM_ENDPOINT_URL"] = REMOTE_LLM_ENDPOINT_URL
if USE_REMOTE_VLM:
    deploy_env["VLM_ENDPOINT_URL"] = REMOTE_VLM_ENDPOINT_URL

# Print the command
display_cmd = " ".join(cmd)
print(f"Command: {display_cmd}")
print(f"Full log: {LOG_FILE}\n")

# --- Phase detection and filtering ---

# Lines matching these patterns are noise — suppress them
SUPPRESS_PATTERNS = [
    re.compile(r"^(\s*[\u2800-\u28FF]|⠋|⠙|⠹|⠸|⠴|⠦|⠧|⠏)"),  # NGC spinner frames
    re.compile(r"^\s*M\u2026|^\s*$"),                              # Truncated NGC progress fragments
    re.compile(r"^\s*#\d+\s+sha256:"),                             # Docker buildkit layer download/extract progress
    re.compile(r"^\s*#\d+\s+extracting\s"),                        # Docker buildkit layer extraction
    re.compile(r"^\s*#\d+\s+\.\.\."),                              # Docker buildkit continuation
    re.compile(r"^time=.*level=warning"),                           # Docker compose unset variable warnings
    re.compile(r"^WARNING! Using --password"),                      # Docker login warning
    re.compile(r"Login Succeeded"),                                 # Docker login success (we print our own)
    re.compile(r"^\s*Getting files to download"),                   # NGC download preamble
    re.compile(r"^\s*━"),                                           # NGC progress bars
    re.compile(r"^\s*[0-9a-f]{12}\s+(Downloading|Extracting|Waiting|Verifying|Pull complete)"),  # Docker layer progress
]

# Lines matching these indicate phase transitions — always show.
# Environment preparation runs in-process above, so these track Compose's own
# lifecycle output.
PHASE_PATTERNS = [
    (re.compile(r"^\s*Network\s+\S+\s+Creat"), "Creating networks"),
    (re.compile(r"^\s*Volume\s+\S+\s+Creat"), "Creating volumes"),
]

# Image pull tracking — service-level "Pulling <name>" / "Pulled <name>"
PULLING_RE = re.compile(r"^\s*Pulling\s+(\S+)")
PULLED_RE = re.compile(r"^\s*Pulled\s+(\S+)")

# Image build tracking — "#N [service-name step/total] COMMAND" / "#N DONE Ns"
BUILD_STEP_RE = re.compile(r"^\s*#\d+\s+\[(\S+)\s+(\d+/\d+)\]")
BUILD_DONE_RE = re.compile(r"^\s*#\d+\s+DONE\s+[\d.]+s")
IMAGE_BUILT_RE = re.compile(r"^\s*Image\s+(\S+)\s+Built")

# Container lifecycle — track creating/starting/healthy
CONTAINER_RE = re.compile(r"^\s*Container\s+(\S+)\s+(Creating|Created|Starting|Started|Healthy|Waiting|Exited.*)")

phases_seen = ["Prepared environment and logged into nvcr.io"]
images_pulling = set()   # images we've seen "Pulling" for
images_pulled = set()    # images we've seen "Pulled" for
builds = {}              # service -> "step/total" for active builds
builds_done = set()      # services that finished building
containers = {}
errors = []
start_time = time.time()

def elapsed():
    s = int(time.time() - start_time)
    return f"{s // 60}m {s % 60:02d}s"

def print_status():
    clear_output(wait=True)
    print(f"Command: {display_cmd}")
    print(f"Full log: {LOG_FILE}\n")

    # Phases
    for p in phases_seen:
        print(f"  [done]  {p}")
    if phases_seen:
        print()

    # Image pull progress
    if images_pulling:
        total = len(images_pulling)
        done = len(images_pulled)
        if done < total:
            still_pulling = sorted(images_pulling - images_pulled)
            print(f"  Pulling images: {done}/{total} complete  ({elapsed()})")
            for img in still_pulling:
                print(f"    {img:<45s} pulling...")
            print()
        else:
            print(f"  Pulling images: {total}/{total} complete\n")

    # Image build progress
    active_builds = {s: step for s, step in builds.items() if s not in builds_done}
    if builds:
        done_count = len(builds_done)
        total_count = len(builds)
        if active_builds:
            print(f"  Building images: {done_count}/{total_count} complete  ({elapsed()})")
            for svc, step in sorted(active_builds.items()):
                print(f"    {svc:<45s} [{step}]")
            print()
        else:
            print(f"  Building images: {total_count}/{total_count} complete\n")

    # Container summary
    if containers:
        healthy = sum(1 for s in containers.values() if s == "Healthy")
        started = sum(1 for s in containers.values() if s in ("Started", "Healthy"))
        total = len(containers)
        print(f"  Containers: {started}/{total} started, {healthy}/{total} healthy  ({elapsed()})")

        # Show containers that aren't healthy yet
        pending = {n: s for n, s in containers.items() if s != "Healthy" and s not in ("Exited",)}
        if pending:
            # Only show non-trivial pending (skip init containers that exited)
            waiting = {n: s for n, s in pending.items() if "Exited" not in s}
            if waiting:
                print()
                for name, status in sorted(waiting.items()):
                    print(f"    {name:<45s} {status}")
        print()

    # Errors
    for e in errors:
        print(f"  ERROR: {e}")

# Run the process. compose_base_args() uses paths relative to deploy/docker,
# so Compose must run from there.
process = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    cwd=DOCKER_DIR,
    env=deploy_env
)

last_refresh = 0
with open(LOG_FILE, "a") as log:
    for line in process.stdout:
        log.write(line)
        log.flush()
        stripped = line.rstrip()

        # Capture errors
        if "[ERROR]" in stripped:
            errors.append(stripped)
            print_status()
            continue

        # Track image pulls (before suppression check)
        m_pulling = PULLING_RE.match(stripped)
        if m_pulling:
            images_pulling.add(m_pulling.group(1))
            now = time.time()
            if now - last_refresh > 2:
                last_refresh = now
                print_status()
            continue

        m_pulled = PULLED_RE.match(stripped)
        if m_pulled:
            images_pulled.add(m_pulled.group(1))
            now = time.time()
            if now - last_refresh > 2:
                last_refresh = now
                print_status()
            continue

        # Track image builds
        m_build = BUILD_STEP_RE.match(stripped)
        if m_build:
            svc, step = m_build.group(1), m_build.group(2)
            builds[svc] = step
            now = time.time()
            if now - last_refresh > 2:
                last_refresh = now
                print_status()
            continue

        m_built = IMAGE_BUILT_RE.match(stripped)
        if m_built:
            svc = m_built.group(1)
            builds_done.add(svc)
            now = time.time()
            if now - last_refresh > 2:
                last_refresh = now
                print_status()
            continue

        # Suppress noise
        if any(p.search(stripped) for p in SUPPRESS_PATTERNS):
            continue

        # Detect phase transitions
        for pattern, label in PHASE_PATTERNS:
            if pattern.search(stripped):
                if label not in phases_seen:
                    phases_seen.append(label)
                    print_status()
                break

        # Track container lifecycle
        m = CONTAINER_RE.match(stripped)
        if m:
            name, status = m.group(1), m.group(2)
            # Normalize "Exited (0) ..." to "Exited"
            if status.startswith("Exited"):
                status = "Exited"
            containers[name] = status
            # Refresh display at most every 2 seconds to avoid flicker
            now = time.time()
            if now - last_refresh > 2:
                last_refresh = now
                print_status()

process.wait()

# Final status
print_status()
print("=" * 50)
if process.returncode == 0 and not errors:
    print(f"Deployment complete in {elapsed()}.")
else:
    print(f"\nDeployment FAILED (exit code {process.returncode}).")
    if errors:
        print(f"\n{len(errors)} error(s) found — see above.")
    print(f"\nFull log: {LOG_FILE}")
    print(f"  View with: cat {LOG_FILE}")

## 9. Verify Deployment

Check that all containers are running and core services are healthy. The health checks poll with retries since some services take a few minutes to fully start.

In [ ]:
import subprocess, time, urllib.request, urllib.error, os

# Show running containers
print("=== Running Containers ===")
subprocess.run(["docker", "ps", "--format", "table {{.Names}}\t{{.Status}}\t{{.Ports}}"])
print()

# Determine proxy port (HAProxy, container: vss-haproxy-ingress)
proxy_port = os.environ.get("PROXY_PORT", "7777")

# Health check endpoints by profile
# Note: HAProxy (vss-haproxy-ingress) has no /health endpoint. The root path "/"
# routes to the UI backend, so checking "/" validates that HAProxy is up and proxying.
checks = [
    ("HAProxy",  f"http://localhost:{proxy_port}/"),
    ("Agent",  "http://localhost:8000/health"),
    ("VST",    "http://localhost:30888/vst/api/v1/sensor/list"),
    ("UI",     "http://localhost:3000"),
]
if PROFILE in ("search", "alerts", "lvs"):
    checks += [
        ("Elasticsearch", "http://localhost:9200"),
        ("Kibana",        "http://localhost:5601/kibana/api/status"),
    ]
if PROFILE == "alerts":
    checks.append(("Video Analytics API", "http://localhost:8081/livez"))

# Poll with retries
MAX_RETRIES = 30
RETRY_INTERVAL = 10
results = {}

print(f"=== Health Checks (up to {MAX_RETRIES * RETRY_INTERVAL}s) ===")
pending = list(checks)

for attempt in range(1, MAX_RETRIES + 1):
    still_pending = []
    for name, url in pending:
        try:
            req = urllib.request.urlopen(url, timeout=5)
            results[name] = f"OK ({req.getcode()})"
        except Exception:
            still_pending.append((name, url))
    pending = still_pending
    if not pending:
        break
    waiting = ", ".join(n for n, _ in pending)
    print(f"  [{attempt}/{MAX_RETRIES}] Waiting for: {waiting}")
    time.sleep(RETRY_INTERVAL)

for name, url in pending:
    results[name] = "FAILED"

print()
all_ok = True
for name, status in results.items():
    marker = "OK" if "OK" in status else "FAIL"
    if marker == "FAIL":
        all_ok = False
    print(f"  {name:.<30s} {status}")

# Check perception container status (no HTTP health endpoint — DeepStream pipeline)
if PROFILE == "search":
    print()
    r = subprocess.run(
        ["docker", "ps", "--filter", "name=vss-rtvi-cv", "--format", "{{.Names}}: {{.Status}}"],
        capture_output=True, text=True
    )
    if r.stdout.strip():
        print("  Perception containers:")
        for line in r.stdout.strip().splitlines():
            print(f"    {line}")
    else:
        print("  WARNING: No vss-rtvi-cv containers found (required for search profile).")
        all_ok = False

print()
if all_ok:
    print("All services healthy.")
else:
    print("Some services failed to start. Check container logs:")
    compose_project_name = os.environ.get("COMPOSE_PROJECT_NAME", "vss")
    print(f"  docker compose -p {compose_project_name} logs <service-name>")

## 10. Access the UI

Once deployment is verified, open the VSS UI in your browser. Accessing the front-end will open up the chat interface where you can interact with the agent and should look like this:

![VSS UI Chat Interface](images/vss_ui_home_page.png)

Run the cell below to generate your VSS UI URL.

**On Brev:** All browser-facing traffic routes through the HAProxy ingress on a single port (default 7777). Create **one** Brev secure link for port 7777 in the dashboard for the primary UI and service proxy. Kibana and Phoenix are accessible via proxy paths (`/kibana` and `/phoenix`) so no separate secure links are needed for those.

**On other cloud providers:** Depending on your CSP's firewall and security group configuration, you may need to expose or forward ports to access the UI and other services from your browser. The following ports are used by VSS:

| Port | Service | Profiles | Brev Secure Link |
|------|---------|----------|------------------|
| 7777 | HAProxy ingress (consolidates UI, Agent, VST) | all | Required (primary) |
| 3000 | VSS UI | all | Not needed (behind proxy) |
| 8000 | VSS Agent API | all | Not needed (behind proxy) |
| 30888 | VST (Video Storage Toolkit) | all | Not needed (behind proxy) |
| 5601 | Kibana | search, alerts, lvs | Not needed (behind proxy at `/kibana`) |
| 6006 | Phoenix (LLM tracing/observability) | all | Not needed (behind proxy at `/phoenix`) |
| 9200 | Elasticsearch | search, alerts, lvs | Not needed |
| 8081 | Video Analytics API | alerts | Not needed |
| 31000 | nvstreamer (WebRTC live view) | search, alerts, lvs | Required for live camera view |
| 8554 | RTSP (if using test stream) | alerts | Not needed |

**Brev summary:** For the **base** profile, create 1 secure link (port 7777). For **search**, **alerts**, and **lvs** profiles, create secure links for ports 7777 and 31000. Kibana and Phoenix are accessible via proxy paths — no separate secure links needed.

If direct access is not possible, use SSH port forwarding or your CSP's port sharing/tunneling feature.

In [ ]:
import os

if BREV_ENV_ID:
    proxy_port = os.environ.get("PROXY_PORT", "7777")
    brev_link_prefix = os.environ.get("BREV_LINK_PREFIX", f"{proxy_port}")
    brev_link_domain = os.environ["BREV_LINK_DOMAIN"]
    ui_url = f"https://{brev_link_prefix}-{BREV_ENV_ID}.{brev_link_domain}"
    print(f"VSS UI (via Brev secure link): {ui_url}")
    print()
    print("Setup:")
    print(f"  1. Ensure a Brev secure link exists for port {proxy_port}")
    print(f"  2. Open: {ui_url}")
    print()
    print("All services (Agent API, VST, UI) are consolidated behind the proxy.")
    print("No individual port forwarding is needed.")
    if PROFILE in ("search", "alerts", "lvs"):
        print()
        print(f"=== Additional Services ({PROFILE}) ===")
        print("These services are accessible via the proxy (no separate secure links needed):")
        print()
        kibana_url = f"{ui_url}/kibana"
        print(f"  Kibana:                   {kibana_url}")
        if PROFILE in ("search", "alerts", "lvs"):
            nvstreamer_url = f"https://31000-{BREV_ENV_ID}.{brev_link_domain}"
            print(f"  nvstreamer (port 31000):  {nvstreamer_url}")
            print("    ^ Requires a separate Brev secure link for port 31000")
        phoenix_url = f"{ui_url}/phoenix"
        print(f"  Phoenix:                  {phoenix_url}  (optional, for LLM tracing)")
else:
    proxy_port = os.environ.get("PROXY_PORT", "7777")
    proxy_url = f"http://{EXTERNAL_IP or HOST_IP}:{proxy_port}"
    ui_url = f"http://{EXTERNAL_IP or HOST_IP}:3000"
    print(f"VSS UI (direct):    {ui_url}")
    print(f"VSS UI (via proxy): {proxy_url}")
    print()
    print("If the URL is not directly accessible, use one of these methods:")
    print()
    print("  SSH port forwarding (works everywhere):")
    print(f"    ssh -L {proxy_port}:localhost:{proxy_port} <user>@{EXTERNAL_IP or HOST_IP}")
    print(f"    Then open: http://localhost:{proxy_port}")
    print()
    print("  VSCode Remote SSH:")
    print("    Connect to the instance via Remote-SSH, ports forward automatically.")
    print()
    if PROFILE in ("search", "alerts", "lvs"):
        print(f"  Kibana (via proxy):  {proxy_url}/kibana")
        print(f"  Kibana (direct):     http://{EXTERNAL_IP or HOST_IP}:5601/kibana")
        if PROFILE in ("search", "alerts", "lvs"):
            print(f"  nvstreamer (live view): http://{EXTERNAL_IP or HOST_IP}:31000")
        print(f"  Phoenix (via proxy): {proxy_url}/phoenix")
        print(f"  Phoenix (direct):    http://{EXTERNAL_IP or HOST_IP}:6006")

## 11. Next Steps

Once you can access the VSS frontend, continue the QuickStart example which involves uploading a video, engaging in Q&A, and generating a report: [Quickstart - Upload a Video](https://docs.nvidia.com/vss/latest/quickstart.html#step-2-upload-a-video)

You can either use your own videos for these examples or download the [VSS Sample Data from NGC](https://docs.nvidia.com/vss/latest/quickstart.html#download-sample-data-from-ngc).

Once you've gone through the QuickStart example, you can follow **Step 12** in this notebook to deploy different [Agent Workflows](https://docs.nvidia.com/vss/latest/agent-workflows.html).

**Step 13** provides instructions on stopping the deployment.

## 12. Profile-Specific Next Steps

Quick-start instructions for your deployed profile.

In [ ]:
if PROFILE == "base":
    print("""=== Base Profile — Quick Start ===

1. Open the VSS UI (see Section 10 for the URL).

2. Upload a video using the "Upload" button in the sidebar.
   Supported formats: MP4, MKV. Wait for the upload to complete.

3. Once uploaded, select the video and start chatting about it.
   Try asking: "What is happening in this video?"

4. The agent uses the VLM to analyze video frames and answers
   questions about the content.
""")

elif PROFILE == "search":
    print("""=== Search Profile — Quick Start ===

1. Open the VSS UI and switch to the "Search" tab.

2. Upload a video using the upload button. The video will be
   split into chunks and embedded for semantic search. This
   takes a few minutes depending on video length.

3. Once processing completes, use the search bar to find moments:
   - "person walking"
   - "red car"
   - "someone carrying a box"

4. Click a search result to play the matching video clip.

5. You can also chat about uploaded videos in the "Chat" tab.

Note: The vss-rtvi-cv container must be running for the embedding
pipeline. Check with: docker ps --filter name=vss-rtvi-cv
""")

elif PROFILE == "alerts":
    print("""=== Alerts Profile — Quick Start ===

1. Open the VSS UI. The alerts profile needs an RTSP camera stream
   to generate detections and alerts.

2. Add a camera sensor:
   - Go to the "Sensors" or camera management section in the UI
   - Add your RTSP stream URL (e.g. rtsp://IP:8554/stream)
   - The perception pipeline will begin analyzing the stream

3. View live detections:
   - Open the "Alerts" tab to see real-time alerts as they're generated
   - Click an alert to view the video clip with bounding boxes

4. Open the "Dashboard" tab to see the Kibana analytics dashboard
   with detection statistics, timelines, and heatmaps.

5. Use the "Chat" tab to ask questions about detected events:
   - "What alerts happened in the last hour?"
   - "How many people were detected today?"
""")

elif PROFILE == "lvs":
    print("""=== LVS Profile — Quick Start ===

1. Open the VSS UI and upload a video via the sidebar.

2. Once uploaded, use the chat to request a report:
   - "Generate a report for my_video.mp4"
   - "Summarize what happens in this video"

3. The agent analyzes the full video and generates a structured
   report with timeline summaries, detected events, and analytics.

4. Reports are saved and accessible via the "Reports" section.
""")

## 13. Stop Deployment

Stop all containers **without deleting data or volumes**. Use this when you want to:
- Free up GPU/memory resources temporarily
- Change to a different profile (update `PROFILE` in Section 1, then re-run from Section 8)
- Restart the deployment later by re-running Section 8

In [ ]:
import subprocess, os

# generated.env only exists after a deploy. Without it Compose would interpolate
# defaults, resolve a different project name and stop nothing.
env_file = profile_path(PROFILE, "generated.env")
if not os.path.isfile(env_file):
    print(f"ERROR: Could not find generated.env at {env_file}")
    print("Has the deployment been run at least once (Section 8)?")
    raise FileNotFoundError(env_file)

compose_project_name = (read_env_value(env_file, "COMPOSE_PROJECT_NAME")
                        or os.environ.get("COMPOSE_PROJECT_NAME", "vss"))
print(f"Using env file: {env_file}")
print(f"Using compose project: {compose_project_name}")
print("Stopping all VSS containers (preserving data and volumes)...\n")

# Compose must see the same three env files as the deploy in Section 8, or
# interpolation falls back to defaults and resolves a different project.
result = subprocess.run(
    compose_base_args(PROFILE) + ["-p", compose_project_name, "stop"],
    capture_output=True, text=True,
    cwd=DOCKER_DIR, env=compose_env(HARDWARE_PROFILE)
)
print(result.stdout)
if result.stderr:
    # Filter out the harmless "variable is not set" warnings
    for line in result.stderr.splitlines():
        if "is not set" not in line:
            print(line)

if result.returncode == 0:
    print("\nAll containers stopped. Re-run Section 8 to start them again.")
else:
    print(f"\nStop exited with code {result.returncode}.")
    print(f"You can also stop manually: docker compose -p {compose_project_name} stop")

## 14. Teardown

Stop all containers and **delete all data** (volumes, models, data directory). Run the cell below when you want to completely remove the deployment.

This brings down every VSS Compose project with `-v --remove-orphans`, deletes the generated env files, removes dangling volumes, clears the sdrc runtime artifacts, and deletes the data directory. If Docker storage was moved to NVMe (Section 4), volume cleanup requires an extra step because Docker can't remove volumes whose data lives outside its data-root (the symlink trick). The cell handles this automatically.

In [ ]:
import subprocess, os, json

# --- Native teardown (Section 7.1 helpers) ---
print("Tearing down VSS deployment...")
teardown_deployment()

# --- Clean up stuck volumes ---
# When Docker's data-root is on NVMe but volumes are symlinked back to root,
# `docker volume rm` fails with "unable to remove a directory outside of the
# local volume root". Fall back to sudo rm for those, then restart Docker.

result = subprocess.run(["docker", "volume", "ls", "-q"], capture_output=True, text=True)
leftover = result.stdout.strip().splitlines()

if leftover:
    print(f"\n{len(leftover)} leftover volume(s). Cleaning up...")
    need_restart = False
    for vol in leftover:
        r = subprocess.run(["docker", "volume", "rm", "-f", vol],
                          capture_output=True, text=True)
        if r.returncode == 0:
            print(f"  removed  {vol}")
        else:
            # Symlinked volume — remove directly from /var/lib/docker/volumes
            vol_path = f"/var/lib/docker/volumes/{vol}"
            r2 = subprocess.run(["sudo", "rm", "-rf", vol_path],
                               capture_output=True, text=True)
            if r2.returncode == 0:
                print(f"  rm'd     {vol}")
                need_restart = True
            else:
                print(f"  FAILED   {vol}: {r2.stderr.strip()}")

    if need_restart:
        print("\n  Restarting Docker to clear volume metadata...")
        subprocess.run(["sudo", "systemctl", "restart", "docker"],
                      capture_output=True, check=True)

    # Verify
    result = subprocess.run(["docker", "volume", "ls", "-q"], capture_output=True, text=True)
    remaining = result.stdout.strip().splitlines()
    if remaining:
        print(f"\n  {len(remaining)} volume(s) still stuck:")
        for v in remaining:
            print(f"    {v}")
    else:
        print("\nAll volumes cleaned up.")
else:
    print("\nAll volumes cleaned up.")

In [ ]:
# To remove the deployment repo from disk:
# import shutil
# shutil.rmtree(REPO_DIR)
# print(f"Removed {REPO_DIR}")